# XRR fitting for Rigaku `.rasx` files (v2: single film or multi-layer stack)

Fits X-ray reflectivity scans of **a thin film, or a stack of layers of different materials, on a substrate** with the Parratt formalism and returns every parameter with its uncertainty.
Version 2 keeps the fitting engine of the earlier notebook unchanged and adds a cleaner way to describe the sample (a layer table) and a structure preview before anything is fitted.

| quantity | notes |
|---|---|
| layer thickness (Å) | statistical error from the fit, plus a systematic error from refitting under different choices; the total thickness of a stack is reported too |
| layer density (g/cm³, % of bulk) | from the electron density; an *apparent* density (see the notes at the end) |
| surface and interface roughness (Å) | Névot–Croce, one value per interface |
| 2θ zero offset, footprint angle, intensity scale, background | nuisance parameters, reported for diagnostics |
| for a stack: **BIC change if a layer is removed** | tells you whether the data actually need each layer (a thin layer can be traded against density and roughness) |

plus a list of quality checks that say whether the number can be trusted or the scan should be repeated. For every scan the notebook exports the full fit details and quality assessment
(parameters with bounds and errors, fit statistics, optimiser information, correlations, residual diagnostics, each check with its measured value, and a readable report) into `xrr_results/`.

**How to use**

1. Run the first cells (imports and functions); nothing needs editing there.
2. Edit the **settings cell (section 1)**: your `.rasx` file(s), the substrate, and the layer structure (a table of layers, listed from the substrate upward).
3. Run the **structure preview** (section 7a). Check that the stack drawn there is the one you mean.
4. Run the fit (section 7b). Results go to `xrr_results/`.

You need Python 3.9+ with `numpy`, `scipy`, `pandas`, `matplotlib` (and `IPython` / Jupyter, which the notebook environment already provides). A first fit takes 1 to 5 minutes (a few more for a stack in `thorough` mode)
because it starts with a global search, so a wrong starting guess cannot trap it in a local minimum.

The last sections contain **self-tests** on simulated data (one film, one stack): they simulate a sample with known parameters, fit it, and show how well they are recovered. Run them once on your machine to check the installation.

## How the fit works

**Data.** Counts versus 2θ from the file (column 2 of `Profile0.txt`), wavelength from the file, $q_z=\frac{4\pi}{\lambda}\sin\frac{2\theta-\theta_0}{2}$, where $\theta_0$ is a fitted zero offset.
Points with zero counts are clipped to 0.5 inside the logarithm.

**Model.** Parratt recursion with Névot–Croce roughness for vacuum / film / substrate, optionally with a second layer
(section 4). The model counts are $I = 10^{s}\,\mathrm{erf}(\theta/\theta_{fp})\,R(q_z)+b$: intensity scale $s$, footprint angle $\theta_{fp}$, background $b$.

**Objective.** The residual of every point is the difference of $\log_{10}$ counts in units of its Poisson uncertainty,

$$r_i=\big[\log_{10}I^{\mathrm{model}}_i-\log_{10}\max(I_i,0.5)\big]\;\sqrt{\max(I_i,1)}\;\ln 10 ,\qquad \chi^2=\sum_i r_i^2 ,$$

because $\sigma(\log_{10}I)=1/(\sqrt{I}\ln 10)$. A log scale is used so that the weak high-angle fringes count as much as the strong low-angle region.

**Optimiser, two stages.**

1. *Global search:* `scipy.optimize.differential_evolution` inside the parameter bounds (population = `POP` x number of parameters, up to `MAXIT` generations; `quick` / `standard` / `thorough` set these). Its role is to find the right basin, since reflectivity fits have many local minima (a local fit started at a wrong thickness stalls there). The two-layer model has a particularly rough landscape, so it is searched from several starts (0, 1 or 3 independent global searches depending on the mode, plus four polishes seeded from the film-only solution with a top layer of 6, 15, 30 and 45 Å) and the lowest $\chi^2$ is kept.
2. *Polish:* `scipy.optimize.least_squares` (trust-region reflective, `x_scale="jac"`) starting from the best point. This is the reported fit.

**Bounds.** Thickness 0.3 to 1.8 times the fringe-spacing estimate $d=2\pi/\Delta q$ (or the values you set); film electron density 0.5 to 1.4 times bulk; film absorption $\beta$ 0 to 0.6 SLD;
both roughnesses 0 to 20 Å; zero offset ±0.1°; footprint angle 0.05° to 5°; angular resolution 0 to 0.05° (2θ); intensity scale from the peak counts to 100 times the peak counts; background 0 to max(5, 5 x noise floor); substrate density ±15%.
Parameters that finish on a bound are flagged, except a value of zero for background, roughness, absorption or resolution, which is physical.

**Fit range.** From the lower limit you set (default 0.2°) up to the last angle where the smoothed counts exceed max(10, 8 x noise floor) (+0.1°). Beyond that a fit of single counts on a log scale is biased.

**Uncertainties.** Statistical: the covariance $(J^TJ)^{-1}$ of the final fit, multiplied by the reduced $\chi^2$ (so a model that does not describe the data inflates the errors instead of hiding behind them).
The total thickness error includes the covariance of film and top layer. Systematic: the fit is repeated with the other model, with the footprint removed and only data above the critical edge, and with the fit range ending 0.6° and 1.0° earlier; the systematic error is the largest deviation of any variant from the main fit.
Both are reported separately; for XRR the systematic error is usually the larger one.

**Model choice.** The film-only model and the two-layer model (film split into a lower and an upper layer) are compared with the Bayesian information criterion, $\mathrm{BIC}=\chi^2+k\ln n$ ($k$ free parameters, $n$ points). The 2-layer model is used only if $\Delta\mathrm{BIC}<-10$ **and** the fitted upper layer is thicker than 3 Å (a layer that collapses to its minimum is not a real layer).

**Quality checks** (each is listed with its measured value and threshold in the export): thickness changes < 5% across the variants; no physical parameter on a bound (zero is allowed for background, roughness, absorption and resolution; the weakly determined footprint angle is only noted); density within 70 to 115% of bulk; fringe-spacing thickness within 15% of the fit;
reduced $\chi^2<15$; more than 3 decades of dynamic range; at least 2 fringe minima; no strong structure in the residuals (lag-1 autocorrelation < 0.6).

**Not modelled.** Anomalous dispersion (the film absorption is free instead), a graded substrate, interlayers, and multilayers with more than a film and one top layer.

In [ ]:
import copy
import glob, io, json, re, time, zipfile, warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, least_squares
from scipy.signal import find_peaks, savgol_filter
from scipy.special import erf
from scipy.ndimage import gaussian_filter1d

plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "Times New Roman", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "axes.unicode_minus": False, "figure.dpi": 110})
warnings.filterwarnings("ignore", category=RuntimeWarning)

# serif text in the notebook itself (Jupyter / JupyterLab; some editors such as VS Code ignore this and use their own font settings)
from IPython.display import HTML, display
display(HTML("<style>"
             ".jp-RenderedHTMLCommon, .text_cell_render, .rendered_html, .jp-OutputArea-output, .output_area pre, .dataframe, .dataframe td, .dataframe th"
             "{font-family: 'Times New Roman', 'STIXGeneral', 'DejaVu Serif', serif !important;}"
             ".jp-RenderedHTMLCommon, .text_cell_render, .rendered_html {font-size: 15px;}"
             "</style>"))
import scipy
print("numpy", np.__version__, "| scipy", scipy.__version__, "| pandas", pd.__version__)

## 1. Settings (edit the next cell)

**Files.** `FILES` is a list of paths or glob patterns; the default fits every `.rasx` next to the notebook. `FILM_ID` names the sample in every results file (left empty, the sample name stored in the `.rasx` file is used).
The scan must be a **specular reflectivity scan (2θ/ω starting near 0°)**.

**Sample structure.** Two ways, both ending in the same report:

* **One film** (optionally split into a lower and an upper layer of the same material by the fit itself, `MODEL = "auto"`): set `FILM`, `FILM_DENSITY`, and leave `PRESET = None`, `LAYERS = None`.
* **A stack of layers of different materials**: list them in `LAYERS` **from the substrate upward** (growth order), one `dict` per layer. Only `name`, `formula` and `density` are required.
  Optional per layer: `t_bounds=(min, max)` in Å (search range of the thickness) and `sld_range=(low, high)` (allowed electron density relative to bulk). Tight bounds help a lot for thin layers.
  A few ready-made examples are in `PRESETS`; set `PRESET = "name"` to use one, or write your own list.

Materials are a chemical formula and a bulk mass density; the parser understands parentheses and fractions (`LaFeO3`, `(La0.7Sr0.3)MnO3`, `MgAl2O4`). Typical bulk densities are listed below the settings; **check them for your material**.

**How many layers?** Every layer adds a thickness, a density, an absorption and a roughness. A reflectivity curve rarely supports more than 2 or 3 layers: use as few as you can justify, and trust a layer only if
its `BIC change if removed` is clearly positive and its thickness is stable across the refits.

In [ ]:
# ------------------------------------------------------------------------------------------------
# DATA
FILES = ["*.rasx"]                       # paths or glob patterns, e.g. ["XRR.rasx"] or ["data/**/XRR*.rasx"]; the default fits every .rasx in the notebook's folder
FILM_ID = ""                             # sample id appended to every results file name, e.g. "H-100626_1-LFO-STO". "" = use the sample name stored in the .rasx file.
                                         # Several scans: {"film1/XRR.rasx": "H-100626_1-LFO-STO", "film3/XRR.rasx": "H-100626_3-LFO-STO"}   (key = path as in FILES, or just the file name)

# SUBSTRATE (always required)
SUBSTRATE = "SrTiO3"                     # formula
SUBSTRATE_DENSITY = 5.12                 # bulk density, g/cm^3

# SAMPLE STRUCTURE ---- choose ONE of the two styles --------------------------------------------------
# (a) one film on the substrate (PRESET = None and LAYERS = None):
FILM = "LaFeO3"                          # film formula
FILM_DENSITY = 6.63                      # bulk density of the film material, g/cm^3
FILM_LATTICE = 3.93                      # optional (A): out-of-plane repeat for a unit-cell count; None to skip

# (b) a stack of layers of DIFFERENT materials, listed from the SUBSTRATE UPWARD. Set PRESET to a name below, or write LAYERS yourself.
PRESETS = {
    # an oxidised metal film on sapphire (calibration films)
    "Fe+Fe2O3 on Al2O3": dict(substrate=("Al2O3", 3.99), layers=[dict(name="Fe", formula="Fe", density=7.874),
                                                                  dict(name="Fe2O3", formula="Fe2O3", density=5.24, t_bounds=(5, 60), sld_range=(0.7, 1.15))]),
    # LaFeO3 on SrTiO3 with a separate interfacial layer (use only if the data need it; check BIC change if removed)
    "LFO with interface layer on STO": dict(substrate=("SrTiO3", 5.12), layers=[dict(name="interface", formula="LaFeO3", density=6.63, t_bounds=(2, 25), sld_range=(0.6, 1.2)),
                                                                              dict(name="LFO", formula="LaFeO3", density=6.63)]),
    # LaFeO3 with a different surface layer (e.g. a La2O3 or Fe2O3-rich cap)
    "LFO with surface cap on STO": dict(substrate=("SrTiO3", 5.12), layers=[dict(name="LFO", formula="LaFeO3", density=6.63),
                                                                          dict(name="cap", formula="La2O3", density=6.51, t_bounds=(2, 30), sld_range=(0.5, 1.2))]),
}
PRESET = None                            # None, or one of the names above (overrides SUBSTRATE / FILM and sets LAYERS)
LAYERS = None                            # or write the list yourself, e.g. [dict(name="Fe", formula="Fe", density=7.874), dict(name="Fe2O3", formula="Fe2O3", density=5.24, t_bounds=(5, 60))]
STACK_COMPARE = True                     # for a stack: also refit without each layer in turn (BIC change says whether every layer is needed)

# FIT OPTIONS -----------------------------------------------------------------------------------
MODEL = "auto"          # one film only: "auto" (film, or film + upper layer if clearly better), "film", or "film+top"
MODE = "thorough"       # "quick" (~1 min, rough), "standard", "thorough" (slowest, most reliable)
FIT_RANGE_2THETA = (0.2, None)           # (low, high) in degrees; None for high = chosen from the noise floor
THICKNESS_BOUNDS = None                  # one film only: (min, max) in A; None = taken from the fringe spacing
FILM_SLD_RANGE = (0.5, 1.4)              # one film only: allowed electron density relative to bulk
SUBSTRATE_BETA_OVER_DELTA = 0.05         # substrate absorption relative to its dispersion (0.02 to 0.1 is typical for oxides)
APPLY_ATTENUATION = False                # multiply counts by the attenuation column of the file (leave False unless you know you need it)
SEED = 0
CHECK_PROFILE = "calibrated"             # quality gates: "calibrated" (chi-square with a systematic-error floor, see the notes at the end) or "strict" (the original noise-only gates)
SYS_FLOOR = 0.05                         # relative intensity error assumed for the instrument and model simplifications (5 %); used by the calibrated profile
OUTPUT_DIR = Path("xrr_results")
RUN_SELFTEST = False                     # set True once to check the installation (section 8, one film)
VERBOSITY = 2                            # 0 = silent, 1 = progress lines, 2 = detailed log of every stage
# ------------------------------------------------------------------------------------------------
if PRESET:
    if PRESET not in PRESETS:
        raise ValueError(f"PRESET '{PRESET}' not in PRESETS: {list(PRESETS)}")
    SUBSTRATE, SUBSTRATE_DENSITY = PRESETS[PRESET]["substrate"]
    LAYERS = copy.deepcopy(PRESETS[PRESET]["layers"])
OUTPUT_DIR.mkdir(exist_ok=True)

Typical bulk densities (g/cm³) for a few materials, for orientation only:
SrTiO₃ 5.12, LaAlO₃ 6.52, MgAl₂O₄ 3.58, Al₂O₃ 3.98, MgO 3.58, Si 2.33, NdGaO₃ 7.57, LaFeO₃ 6.63, LaCoO₃ 7.29, SrRuO₃ 6.49, La₂O₃ 6.51, Co₃O₄ 6.11, Fe₂O₃ 5.24.
Use the density of the phase you expect; the fit reports how far the film deviates from it.

## 2. Materials

In [ ]:
R_E = 2.8179403262e-5     # classical electron radius, A
N_A = 6.02214076e23

# symbol: (Z, atomic mass). Add elements here if you need them.
ELEMENTS = {
    "H": (1, 1.008), "Li": (3, 6.94), "B": (5, 10.81), "C": (6, 12.011), "N": (7, 14.007), "O": (8, 15.999), "F": (9, 18.998),
    "Na": (11, 22.990), "Mg": (12, 24.305), "Al": (13, 26.982), "Si": (14, 28.086), "P": (15, 30.974), "S": (16, 32.06),
    "K": (19, 39.098), "Ca": (20, 40.078), "Sc": (21, 44.956), "Ti": (22, 47.867), "V": (23, 50.942), "Cr": (24, 51.996),
    "Mn": (25, 54.938), "Fe": (26, 55.845), "Co": (27, 58.933), "Ni": (28, 58.693), "Cu": (29, 63.546), "Zn": (30, 65.38),
    "Ga": (31, 69.723), "Ge": (32, 72.630), "As": (33, 74.922), "Sr": (38, 87.62), "Y": (39, 88.906), "Zr": (40, 91.224),
    "Nb": (41, 92.906), "Mo": (42, 95.95), "Ru": (44, 101.07), "Rh": (45, 102.91), "Pd": (46, 106.42), "Ag": (47, 107.87),
    "In": (49, 114.82), "Sn": (50, 118.71), "Ba": (56, 137.33), "La": (57, 138.905), "Ce": (58, 140.12), "Pr": (59, 140.91),
    "Nd": (60, 144.24), "Sm": (62, 150.36), "Eu": (63, 151.96), "Gd": (64, 157.25), "Tb": (65, 158.93), "Dy": (66, 162.50),
    "Ho": (67, 164.93), "Er": (68, 167.26), "Tm": (69, 168.93), "Yb": (70, 173.05), "Lu": (71, 174.97), "Hf": (72, 178.49),
    "Ta": (73, 180.95), "W": (74, 183.84), "Ir": (77, 192.22), "Pt": (78, 195.08), "Au": (79, 196.97), "Pb": (82, 207.2),
    "Bi": (83, 208.98),
}


def parse_formula(formula):
    '''"(La0.7Sr0.3)MnO3" -> {"La": 0.7, "Sr": 0.3, "Mn": 1, "O": 3}'''
    tokens = re.findall(r"([A-Z][a-z]?|\(|\)|\d*\.?\d+)", formula.replace(" ", ""))
    stack = [{}]
    i = 0
    while i < len(tokens):
        tk = tokens[i]
        if tk == "(":
            stack.append({})
        elif tk == ")":
            grp = stack.pop()
            mult = 1.0
            if i + 1 < len(tokens) and re.fullmatch(r"\d*\.?\d+", tokens[i + 1]):
                mult = float(tokens[i + 1]); i += 1
            for el, n in grp.items():
                stack[-1][el] = stack[-1].get(el, 0) + n * mult
        elif tk in ELEMENTS:
            n = 1.0
            if i + 1 < len(tokens) and re.fullmatch(r"\d*\.?\d+", tokens[i + 1]):
                n = float(tokens[i + 1]); i += 1
            stack[-1][tk] = stack[-1].get(tk, 0) + n
        else:
            raise ValueError(f"unknown element or token '{tk}' in formula '{formula}' (add it to ELEMENTS)")
        i += 1
    return stack[0]


def make_material(formula, density):
    comp = parse_formula(formula)
    Z = sum(ELEMENTS[e][0] * n for e, n in comp.items())
    M = sum(ELEMENTS[e][1] * n for e, n in comp.items())
    rho_e = Z * density * N_A / M / 1e24                 # electrons per A^3
    return {"formula": formula, "Z": Z, "M": M, "density": density, "rho_e": rho_e, "sld": R_E * rho_e}   # sld in A^-2


def density_from_sld(mat, sld):
    return sld / R_E * mat["M"] / (mat["Z"] * N_A) * 1e24


film_mat = make_material(FILM, FILM_DENSITY)
sub_mat = make_material(SUBSTRATE, SUBSTRATE_DENSITY)
for m in (film_mat, sub_mat):
    print(f"{m['formula']:>12s}: {m['Z']:.1f} e-/f.u., M = {m['M']:.2f} g/mol, rho = {m['density']} g/cm3, "
          f"electron density {m['rho_e']:.3f} /A^3, SLD = {m['sld'] * 1e5:.3f} x 1e-5 A^-2")

layer_mats = []
if LAYERS:
    layer_mats = [make_material(L["formula"], L["density"]) for L in LAYERS]
    for L, m in zip(LAYERS, layer_mats):
        print(f"layer {L['name']:>8s}: {m['formula']:>8s}, rho = {m['density']} g/cm3, SLD = {m['sld'] * 1e5:.3f} x 1e-5 A^-2")
    print("stack, substrate upward: " + " | ".join([sub_mat["formula"]] + [L["name"] for L in LAYERS]) + " | vacuum")


## 3. Reading the `.rasx` files

In [ ]:
def load_rasx(path, apply_attenuation=False):
    '''Rigaku .rasx (a zip with Data0/Profile0.txt and the measurement-conditions XML).'''
    path = Path(path)
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        prof_name = next((n for n in names if n.endswith("Profile0.txt")), None)
        xml_name = next((n for n in names if n.endswith("MesurementConditions0.xml") or n.endswith("MeasurementConditions0.xml")), None)
        if prof_name is None or xml_name is None:
            raise ValueError(f"{path.name}: not a single-profile rasx file (found {names})")
        prof = z.read(prof_name).decode("utf-8-sig")
        xml = z.read(xml_name).decode("utf-8-sig")
    d = np.loadtxt(io.StringIO(prof), delimiter="\t")
    root = ET.fromstring(xml)
    text = lambda tag: (root.find(f".//{tag}").text if root.find(f".//{tag}") is not None else None)
    scan_axis = next((a.get("Name") for a in root.iter("Axis") if a.get("State") == "Scan"), None)
    out = {
        "path": str(path), "name": path.name, "two_theta": d[:, 0], "counts": d[:, 1].astype(float), "attenuation": d[:, 2] if d.shape[1] > 2 else np.ones(len(d)),
        "wavelength": float(text("WavelengthKalpha1")), "sample": text("SampleName"), "scan_axis": scan_axis,
        "step": float(text("Step")), "speed": float(text("Speed")), "start_time": text("StartTime"),
    }
    out["seconds_per_point"] = 60.0 * out["step"] / out["speed"]
    if apply_attenuation:
        out["counts"] = out["counts"] * out["attenuation"]
    return out


def check_scan(s):
    msgs = []
    if s["scan_axis"] not in ("TwoThetaOmega", "OmegaTwoTheta"):
        msgs.append(f"scan axis is '{s['scan_axis']}', not a 2theta/omega scan: this does not look like XRR")
    if s["two_theta"].min() > 0.5:
        msgs.append(f"scan starts at {s['two_theta'].min():.2f} deg: the critical edge is not covered, thickness is still possible but density/footprint will be poorly determined")
    if np.any(s["attenuation"] != 1):
        msgs.append("attenuation column is not all 1: the attenuator was used. Counts are used as they are in the file; set APPLY_ATTENUATION=True if your intensities show steps")
    return msgs


def ensure_reflectivity(scan):
    # refuse scans that are not specular reflectivity (for example an XRD 2theta/omega scan) instead of fitting nonsense
    t = scan["two_theta"]
    if t.min() > 1.0 or t.max() > 40.0:
        raise ValueError(f"this scan covers 2theta {t.min():.1f}-{t.max():.1f} deg; a reflectivity scan starts near 0 deg and ends below about 40 deg. "
                         "It looks like an XRD scan, not XRR, so it was not fitted")


def expand(patterns):
    files = []
    for p in patterns:
        files += sorted(glob.glob(str(p), recursive=True))
    return [Path(f) for f in dict.fromkeys(files)]

## 4. The reflectivity model

Layers are numbered 0 (vacuum), then the film (and optionally a top layer), then the substrate. Each layer has a scattering-length density
$\rho = \mathrm{SLD} - i\beta$ (in Å⁻², with SLD = $r_e\times$ electron density). With $k_{z,j}=\sqrt{(q_z/2)^2-4\pi(\rho_j-\rho_0)}$ and Névot–Croce roughness,

$$r_{j,j+1}=\frac{k_{z,j}-k_{z,j+1}}{k_{z,j}+k_{z,j+1}}\,e^{-2k_{z,j}k_{z,j+1}\sigma_j^2},\qquad
X_j=\frac{r_{j,j+1}+X_{j+1}e^{2ik_{z,j+1}d_{j+1}}}{1+r_{j,j+1}X_{j+1}e^{2ik_{z,j+1}d_{j+1}}},\quad X_N=0,\quad R=|X_0|^2 .$$

The measured counts are modelled as $I=I_0\,\mathrm{erf}(\theta/\theta_{fp})\,R(q_z)+\text{background}$, where the error function is the **footprint**
correction (the beam is wider than the sample at low angle), and the 2θ axis may be shifted by a small zero offset. The result is smeared with a Gaussian of fitted width
(the **angular resolution**, 0 to 0.05° in 2θ); this rounds off the sharp critical edge, which an ideal-resolution model cannot reproduce.
The fit minimises the difference of $\log_{10}$(counts), weighted by the Poisson noise.

**Layer stack (`LAYERS`).** For layers of different materials (for example Fe and its oxide Fe2O3 on sapphire) the same recursion is used with one layer per entry of `LAYERS`.
Parameters are named per layer from the substrate upward: `t_k` thickness, `sld_k` SLD (bounded around the bulk value of that material), `beta_k` absorption, `sig_k` roughness of the interface
on **top** of layer k (`sig_N` is the surface), plus `sig_sub` (substrate interface) and the global scale, background, zero offset, footprint and resolution.


In [ ]:
def parratt(qz, thickness, rho, sigma):
    '''Reflectivity of a layer stack. rho: complex SLD per layer (vacuum first); thickness: per layer (ignored for first and last); sigma: roughness of each interface.'''
    rho = np.asarray(rho)
    qz = np.asarray(qz, float)
    kz = np.sqrt((qz[None, :] / 2.0) ** 2 - 4 * np.pi * (rho[:, None] - rho[0]) + 0j)
    X = np.zeros(len(qz), dtype=complex)
    for j in range(len(rho) - 2, -1, -1):
        kj, kj1 = kz[j], kz[j + 1]
        r = (kj - kj1) / (kj + kj1) * np.exp(-2 * kj * kj1 * sigma[j] ** 2)
        ph = np.exp(2j * kj1 * thickness[j + 1])
        X = (r + X * ph) / (1 + r * X * ph)
    return np.abs(X) ** 2


def build_spec(nl, film, sub, t_bounds, scale_bounds, bg_max):
    '''Names and bounds of the free parameters (SLD in 1e-5 A^-2).'''
    s0 = film["sld"] * 1e5
    spec = [
        ("t", *t_bounds), ("sld_film", FILM_SLD_RANGE[0] * s0, FILM_SLD_RANGE[1] * s0), ("beta_film", 0.0, 0.6 * s0),
        ("sig_surf", 0.0, 20.0), ("sig_int", 0.0, 20.0), ("log_scale", *scale_bounds), ("bg", 0.0, bg_max),
        ("off", -0.1, 0.1), ("sld_sub", 0.85 * sub["sld"] * 1e5, 1.15 * sub["sld"] * 1e5), ("fp", 0.05, 5.0),
        ("res", 0.0, 0.05),
    ]
    if nl == 2:
        spec[0] = ("t", 3.0, t_bounds[1])                      # the lower layer may be thin: the two layers share the total thickness
        spec += [("t2", 1.0, t_bounds[1]), ("sld2", 0.3 * s0, 1.4 * s0), ("sig_mid", 0.0, 20.0)]
    return [p[0] for p in spec], np.array([p[1:] for p in spec], float)


def model_counts(v, names, two_theta, lam, sub_bod, footprint=True):
    # the counts are smeared with a Gaussian of width "res" (degrees 2theta): the angular resolution of the instrument
    p = dict(zip(names, v))
    res = p.get("res", 0.0)
    x = np.asarray(two_theta, float)
    if res > 1e-4 and len(x) > 3:
        h = float(np.median(np.diff(x)))
        xe = np.arange(x.min() - 4 * res, x.max() + 4 * res + h / 2, h)
        return np.interp(x, xe, gaussian_filter1d(_counts(p, xe, lam, sub_bod, footprint), res / h, mode="nearest"))
    return _counts(p, x, lam, sub_bod, footprint)


def _counts(p, two_theta, lam, sub_bod, footprint):
    th = np.deg2rad((two_theta - p["off"]) / 2)
    q = 4 * np.pi / lam * np.sin(th)
    rs = p["sld_sub"] * 1e-5 * (1 - 1j * sub_bod)
    if "t_1" in p:                                         # stack of layers of different materials: layers 1..N from the substrate upward
        n = 1
        while f"t_{n + 1}" in p:
            n += 1
        rho, thick, sig = [0.0], [0.0], []
        for k in range(n, 0, -1):                          # the beam meets layer N first
            rho.append((p[f"sld_{k}"] - 1j * p[f"beta_{k}"]) * 1e-5); thick.append(p[f"t_{k}"]); sig.append(p[f"sig_{k}"])
        rho.append(rs); thick.append(0.0); sig.append(p["sig_sub"])
        R = parratt(q, thick, rho, sig)
    else:
        rf = (p["sld_film"] - 1j * p["beta_film"]) * 1e-5
        if "t2" in p:
            r2 = (p["sld2"] - 1j * p["beta_film"] * p["sld2"] / p["sld_film"]) * 1e-5
            R = parratt(q, [0, p["t2"], p["t"], 0], [0, r2, rf, rs], [p["sig_surf"], p["sig_mid"], p["sig_int"]])
        else:
            R = parratt(q, [0, p["t"], 0], [0, rf, rs], [p["sig_surf"], p["sig_int"]])
    fp = erf(np.maximum(th, 0) / np.deg2rad(p["fp"] / 2)) if footprint else 1.0
    return 10 ** p["log_scale"] * fp * R + p["bg"]

## 5. The fit

For each file the notebook

1. finds the noise floor and the 2θ range where there is signal (the fit stops where the counts reach the background);
2. estimates the thickness from the fringe spacing (model free) to set the search bounds;
3. fits a **global** search (differential evolution), polished by least squares, for the film model and, if requested, for the two-layer model (film split into a lower and an upper layer);
4. refits under different choices (two-layer model on/off, shorter ranges, no footprint) to get a systematic error;
5. runs quality checks.

In [ ]:
ADVISORY_CHECKS = ("density plausible", "every layer density plausible")      # apparent density depends on the assumed formula and on porosity: shown as a NOTE, not counted as a concern


def _verdict_text(failed, notes):
    if not failed:
        return "PASS: all checks ok" if not notes else f"PASS: no concerns, {len(notes)} note(s)"
    return f"REVIEW: {len(failed)} concern(s)" + (f", {len(notes)} note(s)" if notes else "")


def _chi_check(res, main):
    """Chi-square gate. strict: reduced chi-square from counting noise only < 15. calibrated: chi-square with a relative systematic floor SYS_FLOOR added to the counting variance < 3."""
    if CHECK_PROFILE == "strict":
        return ("reduced chi-square", main["redchi"] < 15, f"{main['redchi']:.2f}", "< 15")
    return (f"reduced chi-square with a {100 * SYS_FLOOR:.0f} % systematic floor", res["chi2_sys"] < 3, f"{res['chi2_sys']:.2f} (counting noise only: {main['redchi']:.1f})", "< 3")


def _structure_check(res):
    """Residual-structure gate. The lag-1 autocorrelation measures how smooth the misfit is, not how large. strict: autocorrelation < 0.6. calibrated: only counted when the misfit also exceeds the systematic floor."""
    if CHECK_PROFILE == "strict":
        return ("no strong structure in the residuals", res["residual_autocorr_lag1"] < 0.6, f"lag-1 autocorrelation {res['residual_autocorr_lag1']:.2f}", "< 0.6")
    bad = res["residual_autocorr_lag1"] >= 0.6 and res["chi2_sys"] >= 3
    return ("no strong structure in the residuals", not bad, f"lag-1 autocorrelation {res['residual_autocorr_lag1']:.2f}; misfit vs the {100 * SYS_FLOOR:.0f} % floor: chi2_sys {res['chi2_sys']:.2f}", "autocorrelation < 0.6 or chi2_sys < 3")


POP, MAXIT = {"quick": (10, 400), "standard": (20, 1500), "thorough": (35, 3000)}[MODE]
N_DE_STARTS = {"quick": 0, "standard": 1, "thorough": 3}[MODE]      # independent global searches for the two-layer model (besides the seeded starts)
ZERO_OK = {"bg", "sig_surf", "sig_int", "sig_mid", "beta_film", "res"}   # a value of zero is physical for these, so the lower bound is not flagged
NUISANCE = {"fp"}      # weakly determined and without influence on the thickness: reported, but a bound hit does not fail the check


def fit_one(tt, I, lam, film, sub, nl, lo, hi, t_bounds, scale_bounds, bg_max, footprint=True, start=None, seed=0):
    names, bounds = build_spec(nl, film, sub, t_bounds, scale_bounds, bg_max)
    m = (tt >= lo) & (tt <= hi)
    x, y = tt[m], I[m]
    w = np.sqrt(np.maximum(y, 1)) * np.log(10)                     # Poisson noise on the log10 scale
    free = [i for i, n in enumerate(names) if footprint or n != "fp"]

    def full(u):
        v = np.empty(len(names)); v[free] = u
        if not footprint:
            v[names.index("fp")] = 1.0
        return v

    def resid(u):
        mod = model_counts(full(u), names, x, lam, SUBSTRATE_BETA_OVER_DELTA, footprint)
        return (np.log10(np.maximum(mod, 1e-3)) - np.log10(np.maximum(y, 0.5))) * w

    de_info = {}
    if start is None:
        de = differential_evolution(lambda u: np.sum(resid(u) ** 2), bounds[free], seed=seed, popsize=POP, maxiter=MAXIT, tol=1e-8, polish=False)
        start = de.x
        de_info = {"de_chi2": float(de.fun), "de_nfev": int(de.nfev), "de_generations": int(de.nit), "de_converged": bool(de.success)}
    ls = least_squares(resid, np.clip(start, bounds[free, 0], bounds[free, 1]), bounds=bounds[free].T, x_scale="jac")
    dof = len(x) - len(free)
    chi2 = float(np.sum(ls.fun ** 2))
    redchi = chi2 / dof
    cov = np.linalg.pinv(ls.jac.T @ ls.jac) * redchi
    err = np.full(len(names), np.nan); err[free] = np.sqrt(np.diag(cov))
    sd = np.sqrt(np.diag(cov)); corr = cov / np.outer(sd, sd)
    v = full(ls.x)
    g = np.zeros(len(free)); g[free.index(names.index("t"))] = 1.0
    if "t2" in names:
        g[free.index(names.index("t2"))] = 1.0
    t_total = v[names.index("t")] + (v[names.index("t2")] if "t2" in names else 0.0)
    return {"v": v, "err": err, "names": names, "nl": nl, "redchi": redchi, "chi2": chi2, "dof": dof, "bic": chi2 + len(free) * np.log(len(x)),
            "x": x, "y": y, "lo": lo, "hi": hi, "footprint": footprint, "u": ls.x, "t_total": float(t_total), "t_total_err": float(np.sqrt(g @ cov @ g)),
            "bounds": bounds, "free_names": [names[i] for i in free], "corr": corr,
            "ls_nfev": int(ls.nfev), "ls_status": int(ls.status), "ls_message": str(ls.message), **de_info,
            "at_bound": [names[i] for i in free if np.isclose(v[i], bounds[i, 1]) or (np.isclose(v[i], bounds[i, 0]) and names[i] not in ZERO_OK)]}


def kiessig_thickness(tt, I, lam, lo, hi):
    '''Model-free thickness from the spacing of the fringe minima in I*q^4: d = 2 pi / dq.'''
    q = 4 * np.pi / lam * np.sin(np.deg2rad(tt / 2))
    m = (tt > max(lo, 0.6)) & (tt < hi)
    if m.sum() < 60:
        return np.nan, np.array([])
    z = np.log10(np.maximum(I[m], 0.5) * q[m] ** 4)
    z = savgol_filter(z, min(51, (m.sum() // 2) * 2 - 1), 2)
    pk, _ = find_peaks(-z, prominence=0.05)
    qm = q[m][pk]
    return (2 * np.pi / np.mean(np.diff(qm)) if len(qm) > 1 else np.nan), qm


def analyze(scan, film, sub, model=MODEL, verbose=1, film_id=""):
    '''verbose: 0 silent, 1 progress lines, 2 detailed log (data summary, range and bounds, optimiser statistics, every refit).'''
    ensure_reflectivity(scan)
    tt, I, lam = scan["two_theta"], scan["counts"], scan["wavelength"]
    t0 = time.time()

    def log(level, msg):
        if verbose >= level:
            print(f"[{time.time() - t0:6.1f} s] {msg}")

    def describe(label, f):
        extra = ""
        if "de_nfev" in f:
            extra = f"; global search {f['de_generations']} generations, {f['de_nfev']} evaluations, {'converged' if f['de_converged'] else 'stopped at the generation limit'}"
        log(2, f"    {label}: chi2 = {f['chi2']:.1f} for {f['dof']} degrees of freedom (reduced {f['redchi']:.2f}), BIC = {f['bic']:.1f}; "
               f"polish {f['ls_nfev']} evaluations, status {f['ls_status']}{extra}")
        log(2, f"    {label}: thickness {f['t_total']:.2f} +/- {f['t_total_err']:.2f} A (statistical); "
               f"parameters on a bound: {', '.join(f['at_bound']) or 'none'}")
    floor = float(np.mean(I[tt > tt.max() - 1.0]))                  # counts in the last degree of the scan
    smooth = savgol_filter(I, 31, 2)
    thr = max(10.0, 8 * floor)
    above = np.where((tt > 1.0) & (smooth > thr))[0]
    signal_end = float(tt[above.max()]) if len(above) else np.nan
    lo = FIT_RANGE_2THETA[0]
    hi = FIT_RANGE_2THETA[1] or (min(signal_end + 0.1, tt.max()) if np.isfinite(signal_end) else min(4.0, tt.max()))
    d_k, qmin = kiessig_thickness(tt, I, lam, lo, hi)
    if THICKNESS_BOUNDS:
        tb = THICKNESS_BOUNDS
    elif np.isfinite(d_k):
        tb = (max(8.0, 0.3 * d_k), min(1500.0, 1.8 * d_k))
    else:
        tb = (10.0, 500.0)
    imax = float(I.max())
    sb = (np.log10(max(imax, 10)) - 0.3, np.log10(max(imax, 10)) + 2.0)
    bgmax = max(5.0, 5 * floor)

    log(2, f"data: {len(tt)} points, 2theta {tt.min():.2f}-{tt.max():.2f} deg, step {scan['step']} deg, {scan['seconds_per_point']:.2f} s/point, "
           f"peak {imax:.3g} counts, noise floor {floor:.2f} counts")
    log(2, f"signal ends near 2theta = {signal_end:.2f} deg (smoothed counts drop below {thr:.1f}); fit range {lo:.2f}-{hi:.2f} deg")
    log(2, f"fringe spacing: {len(qmin)} minima found -> d = {d_k:.1f} A" if np.isfinite(d_k) else "fringe spacing: fewer than 2 minima found, default thickness bounds used")
    log(2, f"bounds: thickness {tb[0]:.0f}-{tb[1]:.0f} A, log10(scale) {sb[0]:.2f}-{sb[1]:.2f}, background 0-{bgmax:.1f} counts, "
           f"film density {FILM_SLD_RANGE[0]:.1f}-{FILM_SLD_RANGE[1]:.1f} x bulk")
    log(2, f"optimiser: {MODE} mode (population {POP} x parameters, up to {MAXIT} generations), seed {SEED}, model option '{model}'")

    kw = dict(film=film, sub=sub, t_bounds=tb, scale_bounds=sb, bg_max=bgmax, seed=SEED)
    fits = {}
    if model in ("auto", "film"):
        log(1, "stage 1: film model (vacuum / film / substrate): global search, then least-squares polish ...")
        fits["film"] = fit_one(tt, I, lam, nl=1, lo=lo, hi=hi, **kw)
        log(1, f"film model done: t = {fits['film']['t_total']:.1f} A, reduced chi2 = {fits['film']['redchi']:.2f}")
        describe("film model", fits["film"])
    if model in ("auto", "film+top"):
        log(1, f"stage 2: two-layer model (film split into a lower and an upper layer): {N_DE_STARTS} global search(es) plus starts seeded from the film-only solution ...")
        cands = []
        if "film" in fits:                                  # seeded starts: split the film-only thickness into film + top layer
            f1 = fits["film"]
            for t2g in (6.0, 15.0, 30.0, 45.0):
                if f1["v"][0] - t2g < 3.0:
                    continue
                s0v = np.r_[f1["v"], t2g, f1["v"][1], 4.0]
                s0v[0] = f1["v"][0] - t2g
                c = fit_one(tt, I, lam, nl=2, lo=lo, hi=hi, start=s0v, **{k: v for k, v in kw.items() if k != "seed"})
                cands.append(("seeded start, top layer %g A" % t2g, c))
        for k_ in range(N_DE_STARTS):
            cands.append((f"global search, seed {SEED + k_}", fit_one(tt, I, lam, nl=2, lo=lo, hi=hi, **{**kw, "seed": SEED + k_})))
        for lab_, c in cands:
            log(2, f"    candidate '{lab_}': chi2 {c['chi2']:.1f}, t = {c['t_total']:.1f} A, top layer {dict(zip(c['names'], c['v']))['t2']:.1f} A")
        fits["film+top"] = min((c for _, c in cands), key=lambda c: c["chi2"])
        log(1, f"two-layer model done: t = {fits['film+top']['t_total']:.1f} A, reduced chi2 = {fits['film+top']['redchi']:.2f}")
        describe("film + upper layer", fits["film+top"])
    d_bic = np.nan
    if len(fits) == 2:
        d_bic = fits["film+top"]["bic"] - fits["film"]["bic"]
        t2_fit = dict(zip(fits["film+top"]["names"], fits["film+top"]["v"]))["t2"]
        use2 = d_bic < -10 and t2_fit > 3.0
        main_key = "film+top" if use2 else "film"
        why = ("the two-layer model is clearly better (delta BIC < -10)" if use2 else
               f"the upper layer collapsed to {t2_fit:.1f} A, so it is not a real layer" if d_bic < -10 else "the extra layer is not justified (delta BIC >= -10)")
        log(1, f"model choice: delta BIC (2-layer minus 1-layer) = {d_bic:.1f} -> {why}; using '{main_key}'")
    else:
        main_key = next(iter(fits))
        log(1, f"model choice: only '{main_key}' was requested")
    main = fits[main_key]

    def refit(lo_, hi_, footprint=True, label=""):
        start = main["u"] if footprint else np.delete(main["v"], main["names"].index("fp"))
        f = fit_one(tt, I, lam, nl=main["nl"], lo=lo_, hi=hi_, footprint=footprint, start=start, **{k: v for k, v in kw.items() if k != "seed"})
        log(2, f"  refit '{label}' (range {lo_:.2f}-{hi_:.2f} deg, footprint {'fitted' if footprint else 'removed'}): t = {f['t_total']:.2f} A, reduced chi2 = {f['redchi']:.2f}")
        return f

    log(1, "refitting under different choices to estimate the systematic error ...")
    variants = {f"main ({main_key})": main}
    if len(fits) == 2:
        variants["other model"] = fits["film" if main_key == "film+top" else "film+top"]
    variants["above the critical edge only (no footprint)"] = refit(max(0.75, lo), hi, footprint=False, label="above the critical edge only")
    for cut in (0.6, 1.0):
        if hi - cut > 2.0:
            variants[f"fit range ends {cut:g} deg earlier"] = refit(lo, hi - cut, label=f"range ends {cut:g} deg earlier")
    names = main["names"]
    p = dict(zip(names, main["v"])); pe = dict(zip(names, main["err"]))

    def spread(getter):
        vals = np.array([getter(f) for f in variants.values()])
        return float(max(getter(main) - vals.min(), vals.max() - getter(main)))

    dens = lambda f: density_from_sld(film, dict(zip(f["names"], f["v"]))["sld_film"] * 1e-5)
    # thickness-weighted mean SLD gives the film + top layer as one density
    def mean_dens(f):
        q_ = dict(zip(f["names"], f["v"]))
        tot = q_["sld_film"] * q_["t"] + (q_["sld2"] * q_["t2"] if "t2" in q_ else 0.0)
        return density_from_sld(film, tot / f["t_total"] * 1e-5)
    qc = lambda sld: 4 * np.sqrt(np.pi * sld * 1e-5)
    two_theta_c = lambda sld: float(2 * np.degrees(np.arcsin(qc(sld) * lam / (4 * np.pi))))

    base_name = f"{Path(scan['name']).stem}_{film_id}" if film_id else Path(scan["name"]).stem
    res = {
        "file": scan["name"], "film_id": film_id or scan["sample"], "base_name": base_name, "sample": scan["sample"], "model": main_key, "delta_bic_2layer_minus_1layer": d_bic,
        "wavelength_A": lam, "fit_range_2theta": (lo, hi), "noise_floor_counts": floor, "signal_end_2theta": signal_end,
        "dynamic_range_decades": float(np.log10(max(imax, 1) / max(floor, 0.5))),
        "thickness_A": main["t_total"], "thickness_stat_err_A": main["t_total_err"], "thickness_sys_err_A": spread(lambda f: f["t_total"]),
        "thickness_kiessig_A": d_k,
        "density_g_cm3": mean_dens(main), "density_film_layer_g_cm3": dens(main), "density_frac_of_bulk": mean_dens(main) / film["density"],
        "density_sys_err_g_cm3": spread(mean_dens),
        "roughness_surface_A": p["sig_surf"], "roughness_surface_stat_err_A": pe["sig_surf"], "roughness_surface_sys_err_A": spread(lambda f: dict(zip(f["names"], f["v"]))["sig_surf"]),
        "roughness_interface_A": p["sig_int"], "roughness_interface_stat_err_A": pe["sig_int"], "roughness_interface_sys_err_A": spread(lambda f: dict(zip(f["names"], f["v"]))["sig_int"]),
        "surface_layer_thickness_A": p.get("t2", 0.0), "film_layer_thickness_A": p["t"],
        "sld_film_1e-5": p["sld_film"], "beta_film_1e-5": p["beta_film"], "sld_substrate_1e-5": p["sld_sub"],
        "two_theta_offset_deg": p["off"], "footprint_angle_deg": p["fp"], "angular_resolution_deg": p["res"], "log10_scale": p["log_scale"], "background_counts": p["bg"],
        "critical_2theta_film_deg": two_theta_c(p["sld_film"]), "critical_2theta_substrate_deg": two_theta_c(p["sld_sub"]),
        "reduced_chi2": main["redchi"], "n_points": len(main["x"]), "params_at_bound": ";".join(main["at_bound"]) or "none",
        "seconds_per_point": scan["seconds_per_point"], "peak_counts": imax, "runtime_s": time.time() - t0,
    }
    if FILM_LATTICE:
        res["n_unit_cells"] = main["t_total"] / FILM_LATTICE

    # residual diagnostics of the main fit
    curve = model_counts(main["v"], names, main["x"], lam, SUBSTRATE_BETA_OVER_DELTA, main["footprint"])
    rn = (np.log10(np.maximum(curve, 1e-3)) - np.log10(np.maximum(main["y"], 0.5))) * np.sqrt(np.maximum(main["y"], 1)) * np.log(10)
    res["residual_rms"] = float(np.sqrt(np.mean(rn ** 2)))
    res["residual_autocorr_lag1"] = float(np.corrcoef(rn[:-1], rn[1:])[0, 1])
    res["residual_fraction_beyond_3sigma"] = float(np.mean(np.abs(rn) > 3))
    _dev = np.log(np.maximum(main["y"], 0.5)) - np.log(np.maximum(curve, 1e-3)); _cnt = np.maximum(main["y"], 1.0)
    res["sys_floor"] = float(SYS_FLOOR); res["chi2_sys"] = float(np.mean(_dev ** 2 / (1.0 / _cnt + SYS_FLOOR ** 2)))   # relative misfit measured against counting noise PLUS the systematic floor

    t_main = main["t_total"]
    rel_sys = 100 * res["thickness_sys_err_A"] / t_main
    rel_fringe = 100 * abs(d_k - t_main) / t_main if np.isfinite(d_k) else np.nan
    spec = [  # name, ok?, measured, rule
        ("thickness stable across the refits", rel_sys < 5, f"{rel_sys:.1f} % (largest deviation of any variant)", "< 5 %"),
        ("no physical parameter on a bound", len([n for n in main["at_bound"] if n not in NUISANCE]) == 0,
         ";".join(n for n in main["at_bound"] if n not in NUISANCE) or "none", "none"),
        ("density plausible", 0.7 < res["density_frac_of_bulk"] < 1.15, f"{100 * res['density_frac_of_bulk']:.0f} % of bulk", "70 to 115 % of bulk"),
        ("fringe-spacing thickness agrees with the fit", (rel_fringe < 15) if np.isfinite(rel_fringe) else None,
         f"{rel_fringe:.1f} % (fringe estimate {d_k:.1f} A)" if np.isfinite(rel_fringe) else "no estimate", "< 15 %"),
        _chi_check(res, main),
        ("dynamic range", res["dynamic_range_decades"] > 3, f"{res['dynamic_range_decades']:.1f} decades", "> 3 decades"),
        ("fringes visible", len(qmin) >= 2, f"{len(qmin)} minima", ">= 2 minima"),
        _structure_check(res),
    ]
    spec = [(n, (None if ok is None else bool(ok)), m_, r_) for n, ok, m_, r_ in spec]      # _plain_bool: np.bool_ False is not `False`, which hid failed checks as n/a
    res["check_details"] = [{"check": n, "status": ("n/a" if ok is None else "ok" if ok else ("NOTE" if n in ADVISORY_CHECKS else "NOT OK")), "measured": m_, "rule": r_} for n, ok, m_, r_ in spec]
    checks = {n: ok for n, ok, _, _ in spec}
    res["checks"] = checks
    failed = [n for n, ok in checks.items() if ok is False and n not in ADVISORY_CHECKS]; notes = [n for n, ok in checks.items() if ok is False and n in ADVISORY_CHECKS]
    res["checks_failed"] = ";".join(failed) or "none"
    res["verdict"] = _verdict_text(failed, notes); res["checks_notes"] = ";".join(notes) or "none"
    advice = []
    if checks["fringes visible"] is False:
        advice.append("fewer than 2 fringes: the film may be very thin or very rough; the thickness then rests on the shape of the curve and is less certain")
    if checks["no physical parameter on a bound"] is False:
        advice.append(f"parameter(s) at a bound ({';'.join(n for n in main['at_bound'] if n not in NUISANCE)}): widen the bounds or the model is missing something")
    if "fp" in main["at_bound"]:
        advice.append("note: the footprint angle is at its bound; it is a weakly determined nuisance parameter and does not change the thickness (compare the variants table)")
    if checks["dynamic range"] is False:
        advice.append("short dynamic range: count longer / use a slower speed")
    if np.isfinite(signal_end) and signal_end > tt.max() - 0.3:
        advice.append("signal still above the noise at the end of the scan: extend the scan to higher 2theta")
    if checks["thickness stable across the refits"] is False:
        advice.append("thickness depends on the model or range: quote it with the full uncertainty, or repeat the scan with more counts at high angle")
    if checks["no strong structure in the residuals"] is False:
        advice.append("the residuals have structure: the model is missing something (graded layers, an interlayer); treat the roughness and density with caution")
    res["advice"] = advice
    settings = {"mode": MODE, "popsize_factor": POP, "max_generations": MAXIT, "seed": SEED, "model_option": model, "fit_range_2theta": [lo, hi],
                "thickness_bounds_A": list(tb), "scale_bounds_log10": list(sb), "background_max": bgmax, "film_sld_range_rel_bulk": list(FILM_SLD_RANGE),
                "substrate_beta_over_delta": SUBSTRATE_BETA_OVER_DELTA, "apply_attenuation": APPLY_ATTENUATION, "film": film["formula"], "film_bulk_density": film["density"],
                "substrate": sub["formula"], "substrate_density": sub["density"],
                "tb_t_bounds": tb, "scale_bounds": sb, "bg_max": bgmax}
    out = {"result": res, "settings": settings, "main": main, "fits": fits, "variants": variants, "scan": scan, "q_minima": qmin, "film": film, "sub": sub,
           "variant_table": pd.DataFrame([{"fit": k, "layers": v["nl"], "thickness_A": v["t_total"], "stat_err_A": v["t_total_err"], "density_g_cm3": dens(v),
                                           "sigma_surface_A": dict(zip(v["names"], v["v"]))["sig_surf"], "sigma_interface_A": dict(zip(v["names"], v["v"]))["sig_int"],
                                           "reduced_chi2": v["redchi"], "BIC": v["bic"]} for k, v in variants.items()])}
    return out

## 5b. Layer stack (layers of different materials)

Used when `LAYERS` is set in the settings cell. Same optimiser, noise model and report as the film model; every layer has its own material (bulk density sets the bounds of its SLD), thickness, absorption and top-interface roughness.
The stack is also refitted **without each layer in turn**: if removing a layer does not raise the BIC by more than 10, that layer is not needed by the data and its thickness should not be trusted.


In [ ]:
def _zero_ok(n):
    '''A value of zero is physical for the absorption, every roughness, the background and the resolution, so a lower-bound hit is not flagged for them.'''
    return n.startswith(("beta_", "sig_")) or n in ("bg", "res")


def build_spec_stack(layers, mats, sub, t_bounds, scale_bounds, bg_max):
    '''Free parameters of a stack of layers of DIFFERENT materials (layers listed from the substrate upward, names t_1..t_N).
    Per layer k: thickness t_k (A), SLD sld_k (1e-5 A^-2), absorption beta_k, roughness sig_k of the interface on TOP of layer k (sig_N = the surface);
    sig_sub = roughness of the substrate interface.  Global: log_scale, bg, off, sld_sub, fp, res.'''
    spec = []
    for k, (L, m) in enumerate(zip(layers, mats), 1):
        s0 = m["sld"] * 1e5
        rlo, rhi = L.get("sld_range", FILM_SLD_RANGE)
        tlo, thi = L.get("t_bounds") or (1.0, t_bounds[1])
        spec += [(f"t_{k}", tlo, thi), (f"sld_{k}", rlo * s0, rhi * s0), (f"beta_{k}", 0.0, 0.6 * s0), (f"sig_{k}", 0.0, 20.0)]
    spec += [("sig_sub", 0.0, 20.0), ("log_scale", *scale_bounds), ("bg", 0.0, bg_max), ("off", -0.1, 0.1),
             ("sld_sub", 0.85 * sub["sld"] * 1e5, 1.15 * sub["sld"] * 1e5), ("fp", 0.05, 5.0), ("res", 0.0, 0.05)]
    return [s[0] for s in spec], np.array([s[1:] for s in spec], float)


def fit_stack(tt, I, lam, layers, mats, sub, lo, hi, t_bounds, scale_bounds, bg_max, footprint=True, start=None, seed=0):
    '''Same optimiser as fit_one (global differential evolution, then trust-region least squares on the Poisson-weighted log10 residual) for a stack of N layers.'''
    names, bounds = build_spec_stack(layers, mats, sub, t_bounds, scale_bounds, bg_max)
    m = (tt >= lo) & (tt <= hi)
    x, y = tt[m], I[m]
    w = np.sqrt(np.maximum(y, 1)) * np.log(10)
    free = [i for i, n in enumerate(names) if footprint or n != "fp"]

    def full(u):
        v = np.empty(len(names)); v[free] = u
        if not footprint:
            v[names.index("fp")] = 1.0
        return v

    def resid(u):
        mod = model_counts(full(u), names, x, lam, SUBSTRATE_BETA_OVER_DELTA, footprint)
        return (np.log10(np.maximum(mod, 1e-3)) - np.log10(np.maximum(y, 0.5))) * w

    de_info = {}
    if start is None:
        de = differential_evolution(lambda u: np.sum(resid(u) ** 2), bounds[free], seed=seed, popsize=POP, maxiter=MAXIT, tol=1e-8, polish=False)
        start = de.x
        de_info = {"de_chi2": float(de.fun), "de_nfev": int(de.nfev), "de_generations": int(de.nit), "de_converged": bool(de.success)}
        # differential evolution tends to settle with the right TOTAL thickness but the wrong split between layers (a local minimum),
        # so also try many random starting points with fast least-squares and keep the best of everything
        best = least_squares(resid, np.clip(start, bounds[free, 0], bounds[free, 1]), bounds=bounds[free].T, x_scale="jac")
        rs = np.random.default_rng(seed + 1000)
        n_ms = {"quick": 30, "standard": 80, "thorough": 200}.get(MODE, 80)
        lob, hib = bounds[free, 0], bounds[free, 1]
        for _ in range(n_ms):
            u0 = lob + rs.random(len(free)) * (hib - lob)
            try:
                trial = least_squares(resid, u0, bounds=bounds[free].T, x_scale="jac", max_nfev=60)
            except Exception:
                continue
            if trial.cost < best.cost:
                best = trial
        de_info["multistart_n"] = n_ms; de_info["multistart_improved"] = bool(best.cost * 2 < de.fun - 1e-6)
        start = best.x
    ls = least_squares(resid, np.clip(start, bounds[free, 0], bounds[free, 1]), bounds=bounds[free].T, x_scale="jac")
    dof = len(x) - len(free)
    chi2 = float(np.sum(ls.fun ** 2)); redchi = chi2 / dof
    cov = np.linalg.pinv(ls.jac.T @ ls.jac) * redchi
    err = np.full(len(names), np.nan); err[free] = np.sqrt(np.diag(cov))
    sd = np.sqrt(np.diag(cov)); corr = cov / np.outer(sd, sd)
    v = full(ls.x)
    tidx = [names.index(f"t_{k}") for k in range(1, len(layers) + 1)]
    g = np.zeros(len(free))
    for i in tidx:
        g[free.index(i)] = 1.0
    return {"v": v, "err": err, "names": names, "nl": len(layers), "redchi": redchi, "chi2": chi2, "dof": dof, "bic": chi2 + len(free) * np.log(len(x)),
            "x": x, "y": y, "lo": lo, "hi": hi, "footprint": footprint, "u": ls.x, "t_total": float(v[tidx].sum()), "t_total_err": float(np.sqrt(g @ cov @ g)),
            "bounds": bounds, "free_names": [names[i] for i in free], "corr": corr,
            "ls_nfev": int(ls.nfev), "ls_status": int(ls.status), "ls_message": str(ls.message), **de_info,
            "at_bound": [names[i] for i in free if np.isclose(v[i], bounds[i, 1]) or (np.isclose(v[i], bounds[i, 0]) and not _zero_ok(names[i]))]}


def analyze_stack(scan, layers, mats, sub, verbose=1, film_id="", compare=True):
    '''Fit a stack of layers of different materials (e.g. Fe + Fe2O3 on sapphire; layers listed from the substrate upward). Returns the same kind of result as analyze().'''
    ensure_reflectivity(scan)
    tt, I, lam = scan["two_theta"], scan["counts"], scan["wavelength"]
    t0 = time.time()
    N = len(layers)

    def log(level, msg):
        if verbose >= level:
            print(f"[{time.time() - t0:6.1f} s] {msg}")
    floor = float(np.mean(I[tt > tt.max() - 1.0]))
    smooth = savgol_filter(I, 31, 2)
    thr = max(10.0, 8 * floor)
    above = np.where((tt > 1.0) & (smooth > thr))[0]
    signal_end = float(tt[above.max()]) if len(above) else np.nan
    lo = FIT_RANGE_2THETA[0]
    hi = FIT_RANGE_2THETA[1] or (min(signal_end + 0.1, tt.max()) if np.isfinite(signal_end) else min(4.0, tt.max()))
    d_k, qmin = kiessig_thickness(tt, I, lam, lo, hi)
    tb = THICKNESS_BOUNDS or ((max(8.0, 0.3 * d_k), min(1500.0, 1.8 * d_k)) if np.isfinite(d_k) else (10.0, 500.0))
    imax = float(I.max())
    sb = (np.log10(max(imax, 10)) - 0.3, np.log10(max(imax, 10)) + 2.0)
    bgmax = max(5.0, 5 * floor)
    log(2, f"data: {len(tt)} points, 2theta {tt.min():.2f}-{tt.max():.2f} deg, peak {imax:.3g} counts, noise floor {floor:.2f}; signal ends near {signal_end:.2f} deg; fit range {lo:.2f}-{hi:.2f} deg")
    log(2, f"fringe spacing -> total thickness {d_k:.1f} A" if np.isfinite(d_k) else "fringe spacing: fewer than 2 minima found, default thickness bounds used")
    log(1, f"stack (substrate upward): {' / '.join(L['name'] for L in layers)} on {sub['formula']}; {len(build_spec_stack(layers, mats, sub, tb, sb, bgmax)[0])} parameters, mode {MODE}")
    kw = dict(sub=sub, t_bounds=tb, scale_bounds=sb, bg_max=bgmax)
    fits = {}
    log(1, "stage 1: full stack, global search then least-squares polish ...")
    fits["full stack"] = main = fit_stack(tt, I, lam, layers, mats, lo=lo, hi=hi, seed=SEED, **kw)
    log(1, f"full stack done: total t = {main['t_total']:.1f} A, reduced chi2 = {main['redchi']:.2f}, BIC {main['bic']:.1f}; on a bound: {', '.join(main['at_bound']) or 'none'}")
    dropped = {}
    if compare and N > 1:
        log(1, "stage 2: the stack without each layer in turn (is every layer needed?) ...")
        for k in range(N):
            lab = f"without {layers[k]['name']}"
            fits[lab] = fit_stack(tt, I, lam, layers[:k] + layers[k + 1:], mats[:k] + mats[k + 1:], lo=lo, hi=hi, seed=SEED, **kw)
            dropped[layers[k]["name"]] = fits[lab]["bic"] - main["bic"]
            log(1, f"  {lab}: reduced chi2 = {fits[lab]['redchi']:.2f}, BIC {fits[lab]['bic']:.1f} (BIC change if the layer is removed: {dropped[layers[k]['name']]:+.1f})")

    def refit(lo_, hi_, footprint=True, label=""):
        start = main["u"] if footprint else np.delete(main["v"], main["names"].index("fp"))
        f = fit_stack(tt, I, lam, layers, mats, lo=lo_, hi=hi_, footprint=footprint, start=start, **kw)
        log(2, f"  refit '{label}' (range {lo_:.2f}-{hi_:.2f} deg, footprint {'fitted' if footprint else 'removed'}): total t = {f['t_total']:.2f} A, reduced chi2 = {f['redchi']:.2f}")
        return f
    log(1, "refitting under different choices to estimate the systematic error ...")
    same = {"main (full stack)": main, "above the critical edge only (no footprint)": refit(max(0.75, lo), hi, footprint=False, label="above the critical edge only")}
    for cut in (0.6, 1.0):
        if hi - cut > 2.0:
            same[f"fit range ends {cut:g} deg earlier"] = refit(lo, hi - cut, label=f"range ends {cut:g} deg earlier")
    variants = dict(same)
    for lab, f in fits.items():
        if lab != "full stack":
            variants[f"{lab} (different model)"] = f
    P = lambda f: dict(zip(f["names"], f["v"]))
    spread = lambda getter: float(max(getter(main) - min(getter(f) for f in same.values()), max(getter(f) for f in same.values()) - getter(main)))
    p, pe = P(main), dict(zip(main["names"], main["err"]))
    layer_rows = []
    for k, (L, m) in enumerate(zip(layers, mats), 1):
        dens = lambda f, k=k, m=m: density_from_sld(m, P(f)[f"sld_{k}"] * 1e-5)
        sld_err = pe[f"sld_{k}"]
        layer_rows.append({"layer": k, "name": L["name"], "formula": m["formula"], "bulk_density_g_cm3": m["density"],
                           "thickness_A": p[f"t_{k}"], "thickness_stat_err_A": pe[f"t_{k}"], "thickness_sys_err_A": spread(lambda f, k=k: P(f)[f"t_{k}"]),
                           "density_g_cm3": dens(main), "density_stat_err_g_cm3": density_from_sld(m, sld_err * 1e-5) if np.isfinite(sld_err) else np.nan,
                           "density_sys_err_g_cm3": spread(dens), "density_frac_of_bulk": dens(main) / m["density"],
                           "sld_1e-5": p[f"sld_{k}"], "beta_1e-5": p[f"beta_{k}"], "roughness_top_interface_A": p[f"sig_{k}"], "roughness_top_stat_err_A": pe[f"sig_{k}"],
                           "roughness_top_sys_err_A": spread(lambda f, k=k: P(f)[f"sig_{k}"]), "bic_change_if_removed": dropped.get(L["name"], np.nan)})
    wfrac = sum(r["thickness_A"] * r["density_frac_of_bulk"] for r in layer_rows) / main["t_total"]
    qc = lambda sld: 4 * np.sqrt(np.pi * sld * 1e-5)
    two_theta_c = lambda sld: float(2 * np.degrees(np.arcsin(qc(sld) * lam / (4 * np.pi))))
    base_name = f"{Path(scan['name']).stem}_{film_id}" if film_id else Path(scan["name"]).stem
    res = {
        "file": scan["name"], "film_id": film_id or scan["sample"], "base_name": base_name, "sample": scan["sample"], "model": "stack", "stack": " / ".join(L["name"] for L in layers) + f" on {sub['formula']}",
        "delta_bic_2layer_minus_1layer": np.nan, "wavelength_A": lam, "fit_range_2theta": (lo, hi), "noise_floor_counts": floor, "signal_end_2theta": signal_end,
        "dynamic_range_decades": float(np.log10(max(imax, 1) / max(floor, 0.5))),
        "thickness_A": main["t_total"], "thickness_stat_err_A": main["t_total_err"], "thickness_sys_err_A": spread(lambda f: f["t_total"]), "thickness_kiessig_A": d_k,
        "density_g_cm3": np.nan, "density_film_layer_g_cm3": layer_rows[0]["density_g_cm3"], "density_frac_of_bulk": wfrac, "density_sys_err_g_cm3": np.nan,
        "roughness_surface_A": p[f"sig_{N}"], "roughness_surface_stat_err_A": pe[f"sig_{N}"], "roughness_surface_sys_err_A": spread(lambda f: P(f)[f"sig_{N}"]),
        "roughness_interface_A": p["sig_sub"], "roughness_interface_stat_err_A": pe["sig_sub"], "roughness_interface_sys_err_A": spread(lambda f: P(f)["sig_sub"]),
        "surface_layer_thickness_A": p[f"t_{N}"], "film_layer_thickness_A": p["t_1"],
        "sld_film_1e-5": p["sld_1"], "beta_film_1e-5": p["beta_1"], "sld_substrate_1e-5": p["sld_sub"], "two_theta_offset_deg": p["off"], "footprint_angle_deg": p["fp"],
        "angular_resolution_deg": p["res"], "log10_scale": p["log_scale"], "background_counts": p["bg"],
        "critical_2theta_film_deg": two_theta_c(p["sld_1"]), "critical_2theta_substrate_deg": two_theta_c(p["sld_sub"]),
        "reduced_chi2": main["redchi"], "n_points": len(main["x"]), "params_at_bound": ";".join(main["at_bound"]) or "none",
        "seconds_per_point": scan["seconds_per_point"], "peak_counts": imax, "runtime_s": time.time() - t0, "layers": layer_rows,
    }
    for r_ in layer_rows:                                   # flat per-layer columns for the summary table
        pre = f"layer{r_['layer']}_{r_['name']}"
        res.update({f"{pre}_thickness_A": r_["thickness_A"], f"{pre}_thickness_stat_err_A": r_["thickness_stat_err_A"], f"{pre}_thickness_sys_err_A": r_["thickness_sys_err_A"],
                    f"{pre}_density_g_cm3": r_["density_g_cm3"], f"{pre}_density_frac_of_bulk": r_["density_frac_of_bulk"], f"{pre}_roughness_top_A": r_["roughness_top_interface_A"]})
    curve = model_counts(main["v"], main["names"], main["x"], lam, SUBSTRATE_BETA_OVER_DELTA, main["footprint"])
    rn = (np.log10(np.maximum(curve, 1e-3)) - np.log10(np.maximum(main["y"], 0.5))) * np.sqrt(np.maximum(main["y"], 1)) * np.log(10)
    res["residual_rms"] = float(np.sqrt(np.mean(rn ** 2))); res["residual_autocorr_lag1"] = float(np.corrcoef(rn[:-1], rn[1:])[0, 1]); res["residual_fraction_beyond_3sigma"] = float(np.mean(np.abs(rn) > 3))
    _dev = np.log(np.maximum(main["y"], 0.5)) - np.log(np.maximum(curve, 1e-3)); _cnt = np.maximum(main["y"], 1.0)
    res["sys_floor"] = float(SYS_FLOOR); res["chi2_sys"] = float(np.mean(_dev ** 2 / (1.0 / _cnt + SYS_FLOOR ** 2)))   # relative misfit measured against counting noise PLUS the systematic floor
    t_main = main["t_total"]
    rel_sys = 100 * res["thickness_sys_err_A"] / t_main
    rel_layer = max(100 * r_["thickness_sys_err_A"] / r_["thickness_A"] for r_ in layer_rows)
    rel_fringe = 100 * abs(d_k - t_main) / t_main if np.isfinite(d_k) else np.nan
    unjust = [r_["name"] for r_ in layer_rows if np.isfinite(r_["bic_change_if_removed"]) and r_["bic_change_if_removed"] < 10]
    bad_dens = [f"{r_['name']} {100 * r_['density_frac_of_bulk']:.0f} %" for r_ in layer_rows if not 0.7 < r_["density_frac_of_bulk"] < 1.15]
    spec_ = [
        ("total thickness stable across the refits", rel_sys < 5, f"{rel_sys:.1f} % (largest deviation of any variant)", "< 5 %"),
        ("every layer thickness stable across the refits", rel_layer < 15, f"{rel_layer:.1f} % (worst layer)", "< 15 %"),
        ("no physical parameter on a bound", len([n for n in main["at_bound"] if n not in NUISANCE]) == 0, ";".join(n for n in main["at_bound"] if n not in NUISANCE) or "none", "none"),
        ("every layer density plausible", not bad_dens, "; ".join(bad_dens) or "all layers 70-115 % of their bulk density", "70 to 115 % of bulk"),
        ("every layer needed (removing it raises the BIC)", (not unjust) if (compare and N > 1) else None, ", ".join(unjust) or ("all layers justified" if (compare and N > 1) else "n/a"), "BIC rises by > 10"),
        ("fringe-spacing thickness agrees with the fit", (rel_fringe < 15) if np.isfinite(rel_fringe) else None, f"{rel_fringe:.1f} % (fringe estimate {d_k:.1f} A)" if np.isfinite(rel_fringe) else "no estimate", "< 15 %"),
        _chi_check(res, main),
        ("dynamic range", res["dynamic_range_decades"] > 3, f"{res['dynamic_range_decades']:.1f} decades", "> 3 decades"),
        ("fringes visible", len(qmin) >= 2, f"{len(qmin)} minima", ">= 2 minima"),
        _structure_check(res),
    ]
    spec_ = [(n, (None if ok is None else bool(ok)), m_, r_) for n, ok, m_, r_ in spec_]      # _plain_bool: np.bool_ False is not `False`, which hid failed checks as n/a
    res["check_details"] = [{"check": n, "status": ("n/a" if ok is None else "ok" if ok else ("NOTE" if n in ADVISORY_CHECKS else "NOT OK")), "measured": m_, "rule": r_} for n, ok, m_, r_ in spec_]
    checks = {n: ok for n, ok, _, _ in spec_}; res["checks"] = checks
    failed = [n for n, ok in checks.items() if ok is False and n not in ADVISORY_CHECKS]; notes = [n for n, ok in checks.items() if ok is False and n in ADVISORY_CHECKS]; res["checks_failed"] = ";".join(failed) or "none"
    res["verdict"] = _verdict_text(failed, notes); res["checks_notes"] = ";".join(notes) or "none"
    advice = []
    if unjust:
        advice.append(f"layer(s) {', '.join(unjust)} are not clearly needed (removing them does not raise the BIC by more than 10): their thickness is not reliable, and a thin layer can be traded against density or roughness")
    if checks["no physical parameter on a bound"] is False:
        advice.append(f"parameter(s) at a bound ({';'.join(n for n in main['at_bound'] if n not in NUISANCE)}): widen the bounds (t_bounds / sld_range per layer) or the model is missing something")
    if checks["every layer thickness stable across the refits"] is False:
        advice.append("a layer thickness depends on the fit range or footprint treatment: a thin layer next to a thick one is weakly constrained; quote it with the full uncertainty or fix its density")
    if checks["fringes visible"] is False:
        advice.append("fewer than 2 fringes: the stack may be thin or rough; the thicknesses then rest on the shape of the curve")
    if "fp" in main["at_bound"]:
        advice.append("note: the footprint angle is at its bound; it is a weakly determined nuisance parameter (compare the variants table)")
    if checks["no strong structure in the residuals"] is False:
        advice.append("the residuals have structure: the model is missing something (an interlayer, a graded oxide); treat roughness and density with caution")
    res["advice"] = advice
    settings = {"mode": MODE, "popsize_factor": POP, "max_generations": MAXIT, "seed": SEED, "model_option": "stack", "fit_range_2theta": [lo, hi], "thickness_bounds_A": list(tb),
                "scale_bounds_log10": list(sb), "background_max": bgmax, "film_sld_range_rel_bulk": list(FILM_SLD_RANGE), "substrate_beta_over_delta": SUBSTRATE_BETA_OVER_DELTA,
                "apply_attenuation": APPLY_ATTENUATION, "film": " / ".join(L["formula"] for L in layers), "film_bulk_density": layers_density_text(mats), "substrate": sub["formula"],
                "substrate_density": sub["density"], "layers": [{k: v for k, v in L.items()} for L in layers], "tb_t_bounds": tb, "scale_bounds": sb, "bg_max": bgmax}
    vt = pd.DataFrame([{"fit": k, "layers": v["nl"], "thickness_A": v["t_total"], "stat_err_A": v["t_total_err"],
                        "sigma_surface_A": dict(zip(v["names"], v["v"]))[f"sig_{v['nl']}"], "sigma_substrate_interface_A": dict(zip(v["names"], v["v"]))["sig_sub"],
                        "reduced_chi2": v["redchi"], "BIC": v["bic"]} for k, v in variants.items()])
    return {"result": res, "settings": settings, "main": main, "fits": fits, "variants": variants, "scan": scan, "q_minima": qmin, "film": mats[0], "sub": sub, "layer_mats": mats, "layers": layers, "variant_table": vt}


def layers_density_text(mats):
    return ", ".join(f"{m['formula']} {m['density']}" for m in mats)


def analyze_any(scan, verbose=1, film_id=""):
    '''Picks the model from the settings: a stack of different materials when LAYERS is set, otherwise the original film (+ upper layer) model.'''
    if LAYERS:
        return analyze_stack(scan, LAYERS, layer_mats, sub_mat, verbose=verbose, film_id=film_id, compare=STACK_COMPARE)
    return analyze(scan, film_mat, sub_mat, verbose=verbose, film_id=film_id)


## 6. Reporting helpers

In [ ]:
def parameter_table_stack(out):
    r, m = out["result"], out["main"]
    me = dict(zip(m["names"], m["err"]))
    rows = [("total thickness", r["thickness_A"], r["thickness_stat_err_A"], r["thickness_sys_err_A"], "A")]
    for L in r["layers"]:
        n = f"{L['name']} (layer {L['layer']})"
        rows += [(f"{n}: thickness", L["thickness_A"], L["thickness_stat_err_A"], L["thickness_sys_err_A"], "A"),
                 (f"{n}: density", L["density_g_cm3"], L["density_stat_err_g_cm3"], L["density_sys_err_g_cm3"], "g/cm3"),
                 (f"{n}: density / bulk", L["density_frac_of_bulk"], np.nan, L["density_sys_err_g_cm3"] / L["bulk_density_g_cm3"], ""),
                 (f"{n}: roughness of its top interface", L["roughness_top_interface_A"], L["roughness_top_stat_err_A"], L["roughness_top_sys_err_A"], "A"),
                 (f"{n}: absorption beta", L["beta_1e-5"], me[f"beta_{L['layer']}"], np.nan, "1e-5 A^-2"),
                 (f"{n}: BIC change if this layer is removed", L["bic_change_if_removed"], np.nan, np.nan, "")]
    rows += [("substrate interface roughness", r["roughness_interface_A"], r["roughness_interface_stat_err_A"], r["roughness_interface_sys_err_A"], "A"),
             ("2theta zero offset", r["two_theta_offset_deg"], me["off"], np.nan, "deg"), ("footprint angle", r["footprint_angle_deg"], me["fp"], np.nan, "deg"),
             ("angular resolution (Gaussian sigma)", r["angular_resolution_deg"], me["res"], np.nan, "deg 2theta")]
    return pd.DataFrame(rows, columns=["parameter", "value", "stat. error", "syst. error", "unit"])


def _add_profile_inset(ax0, out):
    '''SLD depth profile of a fitted stack (inset): z = 0 at the surface, increasing into the sample.'''
    m, r = out["main"], out["result"]
    p = dict(zip(m["names"], m["v"])); n = r["model"] == "stack" and len(out["layers"])
    bounds, sld, sig = [0.0], [0.0], []
    pos = 0.0
    for k in range(n, 0, -1):
        sld.append(p[f"sld_{k}"]); sig.append(p[f"sig_{k}"]); bounds.append(pos); pos += p[f"t_{k}"]
    sld.append(p["sld_sub"]); sig.append(p["sig_sub"]); bounds.append(pos)
    z = np.linspace(-15, pos + 25, 800); prof = np.zeros_like(z)
    for j in range(len(sld) - 1):
        prof += (sld[j + 1] - sld[j]) * 0.5 * (1 + erf((z - bounds[j + 1]) / (np.sqrt(2) * max(sig[j], 0.3))))
    ins = ax0.inset_axes([0.60, 0.50, 0.38, 0.42]); ins.plot(z, prof, color="tab:blue", lw=1.3)
    edge = [0.0] + list(np.cumsum([p[f"t_{k}"] for k in range(n, 0, -1)]))
    for j, k in enumerate(range(n, 0, -1)):
        ins.text((edge[j] + edge[j + 1]) / 2, ins.get_ylim()[1] * 0.98 if False else prof.max() * 1.02, out["layers"][k - 1]["name"], ha="center", va="bottom", fontsize=7)
    for e in edge:
        ins.axvline(e, color="0.7", lw=0.5)
    ins.set_xlabel("depth (A)", fontsize=7); ins.set_ylabel("SLD (1e-5 A$^{-2}$)", fontsize=7); ins.tick_params(labelsize=6)


def parameter_table(out):
    r, m = out["result"], out["main"]
    if r["model"] == "stack":
        return parameter_table_stack(out)
    rows = [
        ("film thickness (total)", r["thickness_A"], r["thickness_stat_err_A"], r["thickness_sys_err_A"], "A"),
        ("film density (thickness-weighted over all layers)", r["density_g_cm3"], np.nan, r["density_sys_err_g_cm3"], "g/cm3"),
        ("film density / bulk", r["density_frac_of_bulk"], np.nan, r["density_sys_err_g_cm3"] / out["film"]["density"], ""),
        ("surface roughness", r["roughness_surface_A"], r["roughness_surface_stat_err_A"], r["roughness_surface_sys_err_A"], "A"),
        ("film/substrate roughness", r["roughness_interface_A"], r["roughness_interface_stat_err_A"], r["roughness_interface_sys_err_A"], "A"),
    ]
    if r["surface_layer_thickness_A"] > 0:
        rows.append(("  upper layer thickness", r["surface_layer_thickness_A"], np.nan, np.nan, "A"))
        rows.append(("  lower layer thickness", r["film_layer_thickness_A"], np.nan, np.nan, "A"))
    rows += [("2theta zero offset", r["two_theta_offset_deg"], dict(zip(m["names"], m["err"]))["off"], np.nan, "deg"),
             ("footprint angle", r["footprint_angle_deg"], dict(zip(m["names"], m["err"]))["fp"], np.nan, "deg"),
             ("angular resolution (Gaussian sigma)", r["angular_resolution_deg"], dict(zip(m["names"], m["err"]))["res"], np.nan, "deg 2theta"),
             ("film absorption beta", r["beta_film_1e-5"], dict(zip(m["names"], m["err"]))["beta_film"], np.nan, "1e-5 A^-2"),
             ("critical angle (film), 2theta", r["critical_2theta_film_deg"], np.nan, np.nan, "deg")]
    if "n_unit_cells" in r:
        rows.append(("unit cells (bulk repeat)", r["n_unit_cells"], np.nan, np.nan, ""))
    return pd.DataFrame(rows, columns=["parameter", "value", "stat. error", "syst. error", "unit"])


def plot_fit(out, save=None):
    s, m = out["scan"], out["main"]
    lam = s["wavelength"]
    q = lambda a: 4 * np.pi / lam * np.sin(np.deg2rad(a / 2))
    curve = model_counts(m["v"], m["names"], m["x"], lam, SUBSTRATE_BETA_OVER_DELTA, m["footprint"])
    fig, ax = plt.subplots(3, 1, figsize=(8, 9.5), sharex=True, gridspec_kw=dict(height_ratios=[3, 3, 1.2]))
    ax[0].semilogy(q(s["two_theta"]), np.clip(s["counts"], 0.5, None), ".", ms=2, color="0.6", label="data")
    ax[0].semilogy(q(m["x"]), curve, color="crimson", lw=1.5, label="fit")
    ax[0].axvspan(q(m["hi"]), q(s["two_theta"].max()), color="0.5", alpha=0.12, label="not fitted (noise)")
    ax[0].set_ylabel("counts"); ax[0].legend(fontsize=8)
    ax[0].set_title(f"{out['result']['file']}: t = {out['result']['thickness_A']:.1f} \u00c5 ({out['result']['model']})")
    if out["result"]["model"] == "stack":
        _add_profile_inset(ax[0], out)
    ax[1].semilogy(q(m["x"]), m["y"] * q(m["x"]) ** 4, ".", ms=2, color="0.6")
    ax[1].semilogy(q(m["x"]), curve * q(m["x"]) ** 4, color="crimson", lw=1.5)
    for qm in out["q_minima"]:
        ax[1].axvline(qm, color="tab:blue", lw=0.6, ls="--")
    ax[1].set_ylabel(r"$I\,q_z^4$  (fringes; dashed = minima)")
    res = (np.log10(curve) - np.log10(np.maximum(m["y"], 0.5))) * np.sqrt(np.maximum(m["y"], 1)) * np.log(10)
    ax[2].plot(q(m["x"]), res, ".", ms=2, color="0.3"); ax[2].axhline(0, color="crimson", lw=1)
    ax[2].set_ylabel("norm. residual"); ax[2].set_xlabel(r"$q_z$ ($\mathrm{\AA^{-1}}$)")
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=140)
    plt.show()


def show_report(out):
    r = out["result"]
    print(f"=== {r['file']}   (sample name in file: {r['sample']}) ===")
    print(f"model: {r['model']}   fit range 2theta {r['fit_range_2theta'][0]:.2f}-{r['fit_range_2theta'][1]:.2f} deg   reduced chi2 {r['reduced_chi2']:.2f}   "
          f"runtime {r['runtime_s']:.0f} s")
    display(parameter_table(out).round(4))
    print("fit variants (the spread of these gives the systematic error):")
    display(out["variant_table"].round(3))
    print(f"quality: {r['verdict']}   (residual rms {r['residual_rms']:.2f}, lag-1 autocorrelation {r['residual_autocorr_lag1']:.2f})")
    display(pd.DataFrame(r["check_details"]))
    if r["advice"]:
        print("advice:")
        for a in r["advice"]:
            print("  -", a)

In [ ]:
def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o


def export_fit_details(out, outdir):
    # Writes, per scan: <stem>_fit_details.json and _parameters_all_fits.csv (parameters, bounds, errors, fit statistics, optimiser information,
    # settings), <stem>_quality.json and _quality.csv (checks with measured values and thresholds, residual diagnostics, advice) and
    # <stem>_report.md (readable summary). Returns the file names.
    outdir = Path(outdir); outdir.mkdir(exist_ok=True)
    r, st, scan = out["result"], out["settings"], out["scan"]
    stem = r["base_name"]
    rows, stats = [], []
    for label, f in out["variants"].items():
        for i, n in enumerate(f["names"]):
            rows.append({"fit": label, "parameter": n, "value": f["v"][i], "stat_error": f["err"][i], "lower_bound": f["bounds"][i, 0], "upper_bound": f["bounds"][i, 1],
                         "on_bound": n in f["at_bound"], "held_fixed": (n == "fp" and not f["footprint"])})
        stats.append({"fit": label, "layers": f["nl"], "fit_range_2theta": [f["lo"], f["hi"]], "footprint_fitted": f["footprint"], "n_points": len(f["x"]),
                      "n_free_parameters": len(f["free_names"]), "dof": f["dof"], "chi2": f["chi2"], "reduced_chi2": f["redchi"], "BIC": f["bic"],
                      "total_thickness_A": f["t_total"], "total_thickness_stat_err_A": f["t_total_err"], "least_squares_nfev": f["ls_nfev"],
                      "least_squares_status": f["ls_status"], "least_squares_message": f["ls_message"],
                      **{k: f[k] for k in ("de_chi2", "de_nfev", "de_generations", "de_converged") if k in f}})
    pt = pd.DataFrame(rows)
    pt.to_csv(outdir / f"{stem}_parameters_all_fits.csv", index=False)
    m = out["main"]
    iu = np.triu_indices(len(m["free_names"]), 1)
    pairs = sorted(zip(np.abs(m["corr"][iu]), [m["free_names"][i] for i in iu[0]], [m["free_names"][j] for j in iu[1]], m["corr"][iu]), reverse=True)[:8]
    details = {
        "film_id": r["film_id"], "file": r["file"], "sample_name_in_file": r["sample"], "scan": {"n_points": len(scan["two_theta"]), "two_theta_range": [float(scan["two_theta"].min()), float(scan["two_theta"].max())],
                                                                         "step_deg": scan["step"], "speed_deg_per_min": scan["speed"], "seconds_per_point": scan["seconds_per_point"],
                                                                         "wavelength_A": scan["wavelength"], "start_time": scan["start_time"], "peak_counts": r["peak_counts"],
                                                                         "noise_floor_counts": r["noise_floor_counts"]},
        "algorithm": {"objective": "sum of squared residuals of log10(counts), each weighted by its Poisson uncertainty",
                      "stage_1": "scipy differential_evolution (global)", "stage_2": "scipy least_squares, trust-region reflective, x_scale='jac'",
                      "uncertainty": "covariance (J^T J)^-1 scaled by the reduced chi-square; systematic error = largest deviation of the refit variants",
                      "model_selection": "BIC; 2-layer model used if delta BIC < -10", "settings": {k: v for k, v in st.items() if not k.startswith(("tb_", "scale_bounds", "bg_max"))}},
        "materials": {"film": out["film"], "substrate": out["sub"]},
        "selected_model": r["model"], "delta_BIC_2layer_minus_1layer": r["delta_bic_2layer_minus_1layer"],
        "results": {k: v for k, v in r.items() if k not in ("checks", "check_details", "advice")},
        "fit_statistics": stats, "largest_parameter_correlations_main_fit": [{"a": a_, "b": b_, "correlation": c_} for _, a_, b_, c_ in pairs],
        "variants_summary": out["variant_table"].to_dict("records"),
        "software": {"numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__},
    }
    (outdir / f"{stem}_fit_details.json").write_text(json.dumps(_clean(details), indent=2), encoding="utf-8")
    quality = {"film_id": r["film_id"], "file": r["file"], "verdict": r["verdict"], "checks": r["check_details"], "advice": r["advice"],
               "residual_diagnostics": {"rms_of_normalised_residuals": r["residual_rms"], "lag1_autocorrelation": r["residual_autocorr_lag1"],
                                        "fraction_beyond_3_sigma": r["residual_fraction_beyond_3sigma"], "reduced_chi2": r["reduced_chi2"]},
               "scan_quality": {"dynamic_range_decades": r["dynamic_range_decades"], "signal_ends_at_2theta": r["signal_end_2theta"], "noise_floor_counts": r["noise_floor_counts"],
                                "seconds_per_point": r["seconds_per_point"]}}
    (outdir / f"{stem}_quality.json").write_text(json.dumps(_clean(quality), indent=2), encoding="utf-8")
    pd.DataFrame(r["check_details"]).to_csv(outdir / f"{stem}_quality.csv", index=False)
    pt_main = parameter_table(out)
    md_lines = [f"# XRR fit report: {r['film_id']} ({r['file']})", "", f"Sample name in file: {r['sample']}", f"Model: {r['model']} (delta BIC 2-layer minus 1-layer: {r['delta_bic_2layer_minus_1layer']:.1f})"
                if np.isfinite(r["delta_bic_2layer_minus_1layer"]) else f"Model: {r['model']}", f"Fit range: {r['fit_range_2theta'][0]:.2f} to {r['fit_range_2theta'][1]:.2f} deg 2theta, "
                f"{r['n_points']} points, reduced chi-square {r['reduced_chi2']:.2f}", "", "## Parameters", "", "| parameter | value | stat. error | syst. error | unit |", "|---|---|---|---|---|"]
    fmt = lambda x: "" if (isinstance(x, float) and not np.isfinite(x)) else f"{x:.4g}"
    md_lines += [f"| {a_} | {fmt(b_)} | {fmt(c_)} | {fmt(d_)} | {e_} |" for a_, b_, c_, d_, e_ in pt_main.itertuples(index=False)]
    md_lines += ["", "## Quality", "", f"**{r['verdict']}**", "", "| check | status | measured | rule |", "|---|---|---|---|"]
    md_lines += [f"| {c['check']} | {c['status']} | {c['measured']} | {c['rule']} |" for c in r["check_details"]]
    if r["advice"]:
        md_lines += ["", "Advice:"] + [f"- {a_}" for a_ in r["advice"]]
    md_lines += ["", "## Fit variants", "", out["variant_table"].round(3).to_markdown(index=False) if hasattr(out["variant_table"], "to_markdown") and _has_tabulate else out["variant_table"].round(3).to_string(index=False)]
    md_lines += ["", "## Algorithm", "", "Global differential evolution followed by trust-region least squares on the Poisson-weighted residual of log10(counts); "
                 "errors from the covariance scaled by the reduced chi-square; systematic error from the refit variants; model chosen by BIC. "
                 f"Settings: mode {st['mode']}, population factor {st['popsize_factor']}, up to {st['max_generations']} generations, seed {st['seed']}."]
    (outdir / f"{stem}_report.md").write_text("\n".join(md_lines), encoding="utf-8")
    return [f"{stem}_fit_details.json", f"{stem}_parameters_all_fits.csv", f"{stem}_quality.json", f"{stem}_quality.csv", f"{stem}_report.md"]


try:
    import tabulate  # noqa: F401
    _has_tabulate = True
except ImportError:
    _has_tabulate = False

## 7. Run on your files

## 7a. Structure preview (check this before fitting)

Prints the structure the fit will use, from the top (vacuum) to the bottom (substrate), with the bulk density and the allowed electron-density range of each layer, and the starting thickness range.
If the table is not the sample you mean, stop and edit the settings.

In [ ]:
def describe_structure():
    rows = []
    if LAYERS:
        mats = [make_material(L["formula"], L["density"]) for L in LAYERS]
        for L, m in reversed(list(zip(LAYERS, mats))):
            lo, hi = L.get("sld_range", (0.5, 1.4)); tb = L.get("t_bounds", "from fringes")
            rows.append((L["name"], L["formula"], L["density"], f"{lo:.2f} - {hi:.2f}", tb))
        kind = f"stack of {len(LAYERS)} layer(s)"
    else:
        lo, hi = FILM_SLD_RANGE
        rows.append(("film", FILM, FILM_DENSITY, f"{lo:.2f} - {hi:.2f}", THICKNESS_BOUNDS if THICKNESS_BOUNDS else "from fringes"))
        kind = f"one film (model '{MODEL}': an upper layer of the same material is added only if clearly better)"
    t = pd.DataFrame(rows, columns=["layer", "formula", "bulk density (g/cm3)", "allowed density (rel. to bulk)", "thickness bounds (A)"])
    print(f"Sample: {kind}\n")
    print("   vacuum / air")
    display(t)
    print(f"   substrate: {SUBSTRATE} ({SUBSTRATE_DENSITY} g/cm3), absorption beta/delta = {SUBSTRATE_BETA_OVER_DELTA}")
    n_free = 4 * len(LAYERS) if LAYERS else 4
    print(f"\nfit mode '{MODE}', fit range from {FIT_RANGE_2THETA[0]} deg; roughly {n_free + 6} free parameters (4 per layer + nuisance)")
    if LAYERS and len(LAYERS) > 3:
        print("WARNING: more than 3 layers is rarely supported by a reflectivity curve; check 'BIC change if removed' for every layer.")
    return t

_structure = describe_structure()
_files = expand(FILES)
print(f"\n{len(_files)} scan(s) to fit:", ", ".join(p.name for p in _files) if _files else "none found (edit FILES)")

## 7b. Fit

In [ ]:
paths = expand(FILES)
results = {}
t_start = time.time()
if LAYERS:
    print(f"settings: stack {' / '.join(L['name'] for L in LAYERS)} on {SUBSTRATE} ({SUBSTRATE_DENSITY} g/cm3); mode '{MODE}', verbosity {VERBOSITY}; "
          f"fit range from {FIT_RANGE_2THETA[0]} deg; output folder {OUTPUT_DIR}")
else:
    print(f"settings: film {FILM} ({FILM_DENSITY} g/cm3) on {SUBSTRATE} ({SUBSTRATE_DENSITY} g/cm3); model '{MODEL}', mode '{MODE}', verbosity {VERBOSITY}; "
          f"fit range from {FIT_RANGE_2THETA[0]} deg; output folder {OUTPUT_DIR}")
if not paths:
    print("No .rasx files found. Put your scan next to this notebook or edit FILES in the settings cell, e.g. FILES = ['XRD.XRR.rasx'].")
else:
    print(f"{len(paths)} file(s) to fit: " + ", ".join(p.name for p in paths))
for n, pth in enumerate(paths, 1):
    print("\n" + "=" * 100)
    print(f"[{n}/{len(paths)}] {pth.name}")
    print("=" * 100)
    scan = load_rasx(pth, APPLY_ATTENUATION)
    fid = (FILM_ID.get(str(pth)) or FILM_ID.get(pth.name) or "") if isinstance(FILM_ID, dict) else FILM_ID
    fid = fid or scan["sample"] or ""
    fid = re.sub(r"[^A-Za-z0-9._-]+", "_", str(fid)).strip("_")
    if VERBOSITY >= 1:
        print(f"  film id used in the results file names: '{fid}'")
    if VERBOSITY >= 1:
        print(f"  sample name in file: {scan['sample']}; scan axis {scan['scan_axis']}; started {scan['start_time']}; wavelength {scan['wavelength']:.6f} A; "
              f"{len(scan['two_theta'])} points, 2theta {scan['two_theta'].min():.2f}-{scan['two_theta'].max():.2f} deg")
    for msg in check_scan(scan):
        print(f"  note: {msg}")
    if VERBOSITY >= 1:
        print(f"  fitting in '{MODE}' mode; a global search is used, so this takes a few minutes.")
    out_ = analyze_any(scan, verbose=VERBOSITY, film_id=fid)
    key = out_["result"]["base_name"]
    results[key] = out_
    r_ = out_["result"]
    if VERBOSITY >= 1:
        print(f"  fit finished in {r_['runtime_s']:.0f} s: thickness {r_['thickness_A']:.2f} A (stat. {r_['thickness_stat_err_A']:.2f}, syst. {r_['thickness_sys_err_A']:.2f}), "
              f"density {100 * r_['density_frac_of_bulk']:.0f} % of bulk, roughness {r_['roughness_surface_A']:.1f} / {r_['roughness_interface_A']:.1f} A; {r_['verdict']}")
        if r_["model"] == "stack":
            for L_ in r_["layers"]:
                print(f"    {L_['name']:>8s}: {L_['thickness_A']:.2f} A (stat. {L_['thickness_stat_err_A']:.2f}, syst. {L_['thickness_sys_err_A']:.2f}), density {100 * L_['density_frac_of_bulk']:.0f} % of bulk, "
                      f"top-interface roughness {L_['roughness_top_interface_A']:.1f} A, BIC change if removed {L_['bic_change_if_removed']:+.1f}")
        print("  checking: building the report, figure and export files ...")
    print()
    show_report(out_)
    plot_fit(out_, save=OUTPUT_DIR / f"{key}_fit.png")
    written = export_fit_details(out_, OUTPUT_DIR)
    print(f"[{pth.name}] exported: " + ", ".join(written) + f", {key}_fit.png")
if paths:
    print(f"\nall {len(paths)} file(s) done in {time.time() - t_start:.0f} s; files are in {OUTPUT_DIR}")

In [ ]:
if results:
    summary = pd.DataFrame([{k: v for k, v in o["result"].items() if k not in ("checks", "check_details", "advice", "layers")} for o in results.values()]).set_index("base_name")
    cols = ["film_id", "thickness_A", "thickness_stat_err_A", "thickness_sys_err_A", "thickness_kiessig_A", "density_frac_of_bulk", "roughness_surface_A",
            "roughness_interface_A", "surface_layer_thickness_A", "reduced_chi2", "checks_failed"]
    cols = cols + [c for c in summary.columns if c.startswith("layer") and c not in cols]            # per-layer columns appear for stack fits
    cols = [c for c in cols if c in summary.columns]
    display(summary[cols].round(3))
    summary.to_csv(OUTPUT_DIR / "xrr_summary.csv")          # one row per scan: results, errors, fit statistics, quality verdict
    for name, o in results.items():
        o["variant_table"].to_csv(OUTPUT_DIR / f"{name}_variants.csv", index=False)
        parameter_table(o).to_csv(OUTPUT_DIR / f"{name}_parameters.csv", index=False)
    print("saved to", OUTPUT_DIR)

## 8. Self-test on simulated data

Simulates a film with **known** parameters (Poisson noise included), fits it with the same code, and compares. The recovered
thickness should agree with the true value to well under 1 % (the statistical error alone is tiny for noise-free-looking data, so a difference of a few times that error,
still far below the systematic error in the variants table, is normal). A recovered thickness that is off by several percent, or a quality verdict other than PASS,
points to a problem with the installation. Set `RUN_SELFTEST = True` in the settings cell.

In [ ]:
if RUN_SELFTEST:
    truth = dict(t=62.0, sld_film=0.92 * film_mat["sld"] * 1e5, beta_film=0.35 * film_mat["sld"] * 1e5 * 0.5, sig_surf=4.0, sig_int=3.0,
                 log_scale=6.2, bg=1.5, off=0.02, sld_sub=sub_mat["sld"] * 1e5, fp=1.6)
    names0 = list(truth)
    tt = np.arange(0.0, 6.0, 0.004)
    rng = np.random.default_rng(7)
    expected = model_counts(np.array([truth[n] for n in names0]), names0, tt, 1.540593, SUBSTRATE_BETA_OVER_DELTA)
    fake = {"name": "simulated", "sample": "simulated", "two_theta": tt, "counts": rng.poisson(expected).astype(float), "wavelength": 1.540593,
            "seconds_per_point": 0.48, "scan_axis": "TwoThetaOmega", "step": 0.004, "speed": 0.5, "attenuation": np.ones(len(tt)), "start_time": None, "path": ""}
    st = analyze(fake, film_mat, sub_mat, model="film", verbose=VERBOSITY)
    r = st["result"]
    rows = [("thickness (A)", truth["t"], r["thickness_A"], r["thickness_stat_err_A"]),
            ("density / bulk", truth["sld_film"] / (film_mat["sld"] * 1e5), r["density_frac_of_bulk"], np.nan),
            ("surface roughness (A)", truth["sig_surf"], r["roughness_surface_A"], r["roughness_surface_stat_err_A"]),
            ("interface roughness (A)", truth["sig_int"], r["roughness_interface_A"], r["roughness_interface_stat_err_A"]),
            ("2theta offset (deg)", truth["off"], r["two_theta_offset_deg"], dict(zip(st["main"]["names"], st["main"]["err"]))["off"])]
    cmp = pd.DataFrame(rows, columns=["parameter", "true", "fitted", "stat. error"])
    cmp["difference / error"] = (cmp["fitted"] - cmp["true"]) / cmp["stat. error"]
    display(cmp.round(3))
    plot_fit(st)
    print("quality of the simulated fit:", st["result"]["verdict"])
else:
    print("self-test skipped (RUN_SELFTEST = False)")

### Self-test of the layer stack

Simulates **Fe (78 A) + Fe2O3 (28 A) on sapphire** with known parameters, fits it with `analyze_stack`, and compares. Set `RUN_STACK_SELFTEST = True` (slow in `thorough` mode; `standard` is enough).


In [ ]:
RUN_STACK_SELFTEST = False
if RUN_STACK_SELFTEST:
    lay = [dict(name="Fe", formula="Fe", density=7.874), dict(name="Fe2O3", formula="Fe2O3", density=5.24, t_bounds=(5, 60), sld_range=(0.7, 1.15))]
    mats = [make_material(L["formula"], L["density"]) for L in lay]
    sub_t = make_material("Al2O3", 3.99)
    truth = dict(t_1=78.0, sld_1=0.95 * mats[0]["sld"] * 1e5, beta_1=0.2 * mats[0]["sld"] * 1e5, sig_1=4.0, t_2=28.0, sld_2=0.92 * mats[1]["sld"] * 1e5, beta_2=0.1 * mats[1]["sld"] * 1e5, sig_2=5.0,
                 sig_sub=3.0, log_scale=6.4, bg=1.5, off=0.02, sld_sub=sub_t["sld"] * 1e5, fp=1.6, res=0.0)
    names0 = list(truth)
    tt = np.arange(0.0, 6.0, 0.004)
    rng = np.random.default_rng(11)
    expected = model_counts(np.array([truth[n] for n in names0]), names0, tt, 1.540593, SUBSTRATE_BETA_OVER_DELTA)
    fake = {"name": "simulated_stack", "sample": "simulated", "two_theta": tt, "counts": rng.poisson(expected).astype(float), "wavelength": 1.540593,
            "seconds_per_point": 0.48, "scan_axis": "TwoThetaOmega", "step": 0.004, "speed": 0.5, "attenuation": np.ones(len(tt)), "start_time": None, "path": ""}
    st = analyze_stack(fake, lay, mats, sub_t, verbose=VERBOSITY)
    r = st["result"]; L1, L2 = r["layers"]
    rows = [("Fe thickness (A)", truth["t_1"], L1["thickness_A"], L1["thickness_stat_err_A"]), ("Fe2O3 thickness (A)", truth["t_2"], L2["thickness_A"], L2["thickness_stat_err_A"]),
            ("total thickness (A)", truth["t_1"] + truth["t_2"], r["thickness_A"], r["thickness_stat_err_A"]),
            ("Fe density / bulk", truth["sld_1"] / (mats[0]["sld"] * 1e5), L1["density_frac_of_bulk"], np.nan), ("Fe2O3 density / bulk", truth["sld_2"] / (mats[1]["sld"] * 1e5), L2["density_frac_of_bulk"], np.nan),
            ("surface roughness (A)", truth["sig_2"], r["roughness_surface_A"], r["roughness_surface_stat_err_A"]), ("Fe/Fe2O3 roughness (A)", truth["sig_1"], L1["roughness_top_interface_A"], L1["roughness_top_stat_err_A"]),
            ("substrate interface (A)", truth["sig_sub"], r["roughness_interface_A"], r["roughness_interface_stat_err_A"])]
    cmp = pd.DataFrame(rows, columns=["parameter", "true", "fitted", "stat. error"])
    cmp["difference / error"] = (cmp["fitted"] - cmp["true"]) / cmp["stat. error"]
    display(cmp.round(3))
    plot_fit(st)
    print("quality of the simulated stack fit:", r["verdict"])
else:
    print("stack self-test skipped (RUN_STACK_SELFTEST = False)")


## 9. Notes and limits

* **Density is apparent.** XRR measures the electron density. The notebook converts it to a mass density with the formula you gave, so a different composition, voids, or
  an oxygen deficiency all show up as a different "density". Absorption (anomalous dispersion) is not modelled: the film absorption is a free parameter.
* **Thickness** is the most robust result: it depends on the fringe spacing. If fewer than two fringes are visible (very thin, very rough film), the uncertainty is larger than
  the statistical error suggests.
* **Roughness and interface sharpness** are model dependent and correlated with density and the background; use the systematic error column.
* **Footprint.** A free angle `fp` describes the intensity roll-off below the critical angle. If it is pinned at a bound the sample is probably shorter or longer than the beam
  setup assumes; the thickness is not affected, the density can be.
* **Absorption in the substrate** is fixed by `SUBSTRATE_BETA_OVER_DELTA`. Change it between 0.02 and 0.1 to see its effect.
* **Fitting range.** The fit stops where the counts reach the noise floor, because a log-scale fit of single counts is biased. If the signal dies early, count longer at high angle.
* **Several layers.** Either one film with an optional upper layer of the same material (`MODEL`), or a stack of layers of different materials (`LAYERS`, section 5b). A graded interface or an interlayer
  is not modelled separately: add it as a thin layer. A thin layer next to a thick one is weakly constrained and can be traded against its density and roughness; check the `BIC change if removed`
  and the stability of its thickness across the refits before quoting it.
* The scans must be specular. Off-specular or rocking scans are not reflectivity.
* Units: lengths in Å, SLD in 10⁻⁵ Å⁻², angles in degrees, densities in g/cm³.

Questions or a case that does not fit well: send the `.rasx` file and the `xrr_results/*_result.json`.
* **Quality checks (`CHECK_PROFILE`).** The checks come in two kinds. *Reliability* checks test the parameters: stability across refits and across seeds, parameters on a bound, every layer needed, fringe agreement, dynamic range.
  *Fit-quality* checks test whether the curve matches: with 10^5 to 10^7 counts the counting noise is a fraction of a percent, so any 1 to 3 % mismatch the model cannot represent (beam footprint, resolution function, detector linearity) is tens of noise-sigmas.
  The original gates (reduced chi-square from counting noise only below 15, lag-1 residual autocorrelation below 0.6) therefore fail almost every high-count scan, also visually excellent fits (13 of 19 fits in our database fail the autocorrelation gate).
  The `calibrated` profile (default) adds a relative systematic floor `SYS_FLOOR` (5 %) to the counting variance: `chi2_sys` below 3 means the model matches the data to about 5 %. The residual-structure check counts only when the misfit also exceeds that floor.
  On the 19 fits in our database a 5 % floor passes 14 of 16 LFO fits and none of the 3 Fe on sapphire fits whose median deviation is 12 to 24 %; 10 % was too lenient. The raw reduced chi-square is always reported.
  Set `CHECK_PROFILE = "strict"` for the original gates. A pass on fit quality says the curve is matched, not that the layer model is unique: use the seed check for that.
* **Concerns and notes.** A failed reliability or fit-quality check is a *concern* and is counted in the verdict (`REVIEW: n concern(s)`). The density-plausibility check (a layer outside 70 to 115 % of its bulk density) is only a *note*:
  the apparent density depends on the assumed formula and on porosity, so a low value is information, not necessarily an error. Notes are listed but not counted (`PASS: no concerns, 1 note(s)`).
